# 9. Cell contours and movies

**Part A — minimal-bridge contours.** Cell outlines drawn around all tracks of a cell, galleries, raw-movie
stills with tracks and contour, and single-molecule follow views.

**Part B — live state + softmax movies.** One track drawn as it moves, coloured by behaviour state, next to its
softmax over time (and the raw movie). Then a batch of movies to choose from, re-rendering of picked ones,
and the supplementary movies.

Each part loads its own data, so you can start at either.

# Part A — minimal-bridge contours

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

sys.path.insert(0, "..")
from sptnano import config, contours, raw_images

paths = config.load_paths()

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(40)
pl.Config.set_tbl_width_chars(180)

## 1. Paths

In [ ]:
CONTOURS_PARQUET = paths["contours_parquet"]
DATA_DIR = CONTOURS_PARQUET.parent
PLOTS_DIR = paths["plots_dir"]
RESULTS_DIR = PLOTS_DIR / "minimal_bridge_contours_2026"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INSTANT_PARQUET = DATA_DIR / "instant_df_hierarchical_gates.parquet"
WINDOWED_PARQUET = DATA_DIR / "windowed_df_hierarchical_gates.parquet"
CONTOURS_SUMMARY_CSV = DATA_DIR / "contours_df_minimal_bridge_by_cell_key_summary.csv"
INSTANT_WITH_CONTOURS_PARQUET = DATA_DIR / "instant_df_with_contours_by_cell_key.parquet"
WINDOWED_WITH_CONTOURS_PARQUET = DATA_DIR / "windowed_df_with_contours_by_cell_key.parquet"

RAW_MASTER_DIR = None  # None = raw movies are found from the foldername column
FRAME_INTERVAL = config.TIME_BETWEEN_FRAMES
PIXEL_SIZE_UM = config.PIXELSIZE_MICRONS
CONTOUR_COORDINATE_UNITS = "pixels"
CONTOUR_X_COL = "x"
CONTOUR_Y_COL = "y"
TRACK_ID_COL = "unique_id"
CELL_KEY_COL = "cell_key"

# Alignment diagnostic selected this correction for raw/movie overlays.
RAW_TRACK_SPATIAL_TRANSFORM = None

## Fiji LUTs And Raw Display Defaults

In [ ]:
# Fiji LUT loading and raw display defaults.
LOAD_FIJI_LUTS = True
FIJI_LUT_DIR = str(paths["fiji_lut_dir"])
DEFAULT_FIJI_LUT_NAME = "JDM_Ink Wash Yellow"

if LOAD_FIJI_LUTS:
    JDM_LUTS, chosen_LUT = raw_images.load_fiji_lut_collection(
        FIJI_LUT_DIR,
        prefix="JDM",
        default_lut_name=DEFAULT_FIJI_LUT_NAME,
    )
    print(f"Loaded {len(JDM_LUTS)} JDM LUTs")
    print("chosen_LUT:", DEFAULT_FIJI_LUT_NAME)
else:
    JDM_LUTS, chosen_LUT = {}, None

RAW_INVERT = False
RAW_CONTRAST_LIMITS = None  # e.g. (100, 2500), or None for auto scaling
RAW_CMAP = "gray"
RAW_LUT = None  # set to chosen_LUT or JDM_LUTS["JDM_Ink Wash Yellow"]

## 2. Load Saved Contours And Mapped Dataframes

In [ ]:
# Run this after a kernel restart if cell-key-safe contour outputs already exist.
LOAD_SAVED_OUTPUTS = True

if LOAD_SAVED_OUTPUTS:
    required_outputs = [CONTOURS_PARQUET, INSTANT_WITH_CONTOURS_PARQUET, WINDOWED_WITH_CONTOURS_PARQUET]
    missing_outputs = [path for path in required_outputs if not path.exists()]
    if missing_outputs:
        print("Missing cell-key-safe output files. Run the optional contour generation cell once with RUN_CONTOUR_GENERATION = True.")
        for path in missing_outputs:
            print("Missing:", path)
    else:
        contours_df = pl.read_parquet(CONTOURS_PARQUET)
        contours_ok_df = contours_df.filter(pl.col("contour_status") == "ok") if "contour_status" in contours_df.columns else contours_df
        failed_contours_df = contours_df.filter(pl.col("contour_status") != "ok") if "contour_status" in contours_df.columns else pl.DataFrame()

        instant_df_with_contours = pl.read_parquet(INSTANT_WITH_CONTOURS_PARQUET)
        windowed_df_with_contours = pl.read_parquet(WINDOWED_WITH_CONTOURS_PARQUET)
        instant_df = contours.with_cell_key(pl.read_parquet(INSTANT_PARQUET), key_col=CELL_KEY_COL)
        windowed_df = contours.with_cell_key(pl.read_parquet(WINDOWED_PARQUET), key_col=CELL_KEY_COL)

        print(f"contours_df: {contours_df.shape}")
        print(f"successful contours: {contours_ok_df.height}")
        print(f"failed contours: {failed_contours_df.height}")
        print(f"instant_df_with_contours: {instant_df_with_contours.shape}")
        print(f"windowed_df_with_contours: {windowed_df_with_contours.shape}")

## Optional: Generate Contours If Needed

Skip this if the saved parquet files above already exist.

In [ ]:
RUN_CONTOUR_GENERATION = False
RECOMPUTE_CONTOURS = False

CONTOUR_METHOD = "minimal_bridge"
MINIMAL_BRIDGE_POINT_RADIUS = 4.0
MINIMAL_BRIDGE_WIDTH = 3.0
MINIMAL_BRIDGE_MASK_PADDING = 12
CONTOUR_EDGE_ANGLE_BINS = 96
CONTOUR_EDGE_PADDING = 0.0
CONTOUR_EDGE_SMOOTHING = 0.15

IDENTIFIER_COLS = [
    "foldername", "filename", "group", "cell", "type", "mol", "geno",
    "cas", "dox", "hap40status", "nocodazole_concentration", "location",
]

if RUN_CONTOUR_GENERATION:
    instant_df = contours.with_cell_key(pl.read_parquet(INSTANT_PARQUET), key_col=CELL_KEY_COL)
    windowed_df = contours.with_cell_key(pl.read_parquet(WINDOWED_PARQUET), key_col=CELL_KEY_COL)
    metadata_cols = [col for col in IDENTIFIER_COLS if col in instant_df.columns]

    if CONTOURS_PARQUET.exists() and not RECOMPUTE_CONTOURS:
        contours_df = pl.read_parquet(CONTOURS_PARQUET)
    else:
        contours_df = contours.compute_cell_contours_dataframe(
            instant_df,
            group_col=CELL_KEY_COL,
            coordinate_units=CONTOUR_COORDINATE_UNITS,
            x_col=CONTOUR_X_COL,
            y_col=CONTOUR_Y_COL,
            metadata_cols=metadata_cols,
            selection_method=None,
            n_cells=None,
            track_id_col=TRACK_ID_COL,
            contour_method=CONTOUR_METHOD,
            edge_angle_bins=CONTOUR_EDGE_ANGLE_BINS,
            edge_padding=CONTOUR_EDGE_PADDING,
            edge_smoothing=CONTOUR_EDGE_SMOOTHING,
            minimal_bridge_point_radius=MINIMAL_BRIDGE_POINT_RADIUS,
            minimal_bridge_width=MINIMAL_BRIDGE_WIDTH,
            minimal_bridge_mask_padding=MINIMAL_BRIDGE_MASK_PADDING,
            on_error="record",
            show_progress=True,
            return_backend="polars",
        )
        contours_df.write_parquet(CONTOURS_PARQUET)
        summary_cols = [col for col in contours_df.columns if col not in {"contour_x", "contour_y"}]
        contours_df.select(summary_cols).write_csv(CONTOURS_SUMMARY_CSV)

    contours_ok_df = contours_df.filter(pl.col("contour_status") == "ok") if "contour_status" in contours_df.columns else contours_df
    instant_df_with_contours = contours.add_contour_measurements_by_filename(instant_df, contours_ok_df, group_col=CELL_KEY_COL, prefix="contour_")
    windowed_df_with_contours = contours.add_contour_measurements_by_filename(windowed_df, contours_ok_df, group_col=CELL_KEY_COL, prefix="contour_")
    instant_df_with_contours.write_parquet(INSTANT_WITH_CONTOURS_PARQUET)
    windowed_df_with_contours.write_parquet(WINDOWED_WITH_CONTOURS_PARQUET)

    print(f"Saved contours to {CONTOURS_PARQUET}")
    print(f"Saved mapped instant/windowed dataframes to {RESULTS_DIR}")

## 3. Filter To Conditions Of Interest

Edit these lists to define the display dataset. Empty lists mean do not filter that column.

In [ ]:
# =============================================================================
# DATA FILTERING CELL for display dataframes
# =============================================================================

DISPLAY_FILTERS = {
    "mol": ["HTT"],
    "type": ["vent"],
    "location": ["GC"],
    "nocodazole_concentration": ["none"],
    "geno": ["20H20S", "56H20S", "77H20S", "150H20S", "20H56S", "20H77S", "20H150S"],
    "group": ["wtkomt", "HTT77", "HTT56", "HTT150"],
}
# Last used for ES myosin:
# DISPLAY_FILTERS = {"dox": ["NA"], "cas": ["NA"], "type": ["ES"], "mol": ["myosin"],
#                    "nocodazole_concentration": ["none"], "geno": ["RUES2"]}

def apply_display_filters(df, filters):
    out = df
    for col, allowed in filters.items():
        if allowed is None or len(allowed) == 0:
            continue
        if col in out.columns:
            out = out.filter(pl.col(col).is_in(allowed))
        else:
            print(f"Skipping filter {col!r}: column not present")
    return out

visual_instant_df = apply_display_filters(instant_df_with_contours, DISPLAY_FILTERS)
visual_windowed_df = apply_display_filters(windowed_df_with_contours, DISPLAY_FILTERS)
visual_contours_ok_df = apply_display_filters(contours_ok_df, DISPLAY_FILTERS)

print("visual_instant_df:", visual_instant_df.shape)
print("visual_windowed_df:", visual_windowed_df.shape)
print("visual_contours_ok_df:", visual_contours_ok_df.shape)

display(
    visual_windowed_df
    .group_by([col for col in ["mol", "cell", "type", "location", "geno", "group", "cas", "dox", "nocodazole_concentration", "replicate"] if col in visual_windowed_df.columns])
    .agg([
        pl.col("filename").n_unique().alias("n_cells"),
        pl.col(TRACK_ID_COL).n_unique().alias("n_tracks"),
        pl.len().alias("n_datapoints"),
    ])
    .sort([col for col in ["mol", "cell", "type", "location", "geno", "group"] if col in visual_windowed_df.columns])
)

## 4. Gallery QC

In [ ]:
RUN_CONTOUR_GALLERY = True

# Gallery cell and track options.
CONTOUR_GALLERY_N_CELLS = 20
CONTOUR_GALLERY_SELECTION = "most_tracks"  # "most_tracks", "most_points", or "random"
CONTOUR_GALLERY_PRESELECT_TRACKS = True
CONTOUR_GALLERY_TRACK_SELECTION_METHOD = "balanced_by_population"  # "longest", "most_points", "random", "balanced_by_population"
CONTOUR_GALLERY_TARGET_TRACKS_PER_CELL = 10
CONTOUR_GALLERY_TRACKS_PER_POPULATION = 3
CONTOUR_GALLERY_TRACK_MIN_POINTS = 100
CONTOUR_GALLERY_TRACK_MIN_LENGTH_FRAMES = None
CONTOUR_GALLERY_FRAME_RANGE = None
CONTOUR_GALLERY_POPULATION_COL = "final_population"  # e.g. "final_population"
CONTOUR_GALLERY_POPULATIONS = ["bound", "confined", "exploratory", "transport"] #None     # e.g. ["bound", "confined", "exploratory", "transport"]

# Gallery visual options.
CONTOUR_GALLERY_SHOW_RAW_PROJECTION = False
CONTOUR_GALLERY_RAW_PROJECTION = "max"
CONTOUR_GALLERY_RAW_FRAME_RANGE = None
CONTOUR_GALLERY_RAW_MAX_FRAMES = None
CONTOUR_GALLERY_RAW_CONTRAST_LIMITS = RAW_CONTRAST_LIMITS
CONTOUR_GALLERY_INVERT_RAW = RAW_INVERT
CONTOUR_GALLERY_RAW_CMAP = RAW_CMAP
CONTOUR_GALLERY_RAW_LUT = RAW_LUT
CONTOUR_GALLERY_SHOW_TRACKS = True
CONTOUR_GALLERY_TRACK_COLOR_MODE = "single"  # "by_contour", "by_track", or "single"
CONTOUR_GALLERY_SINGLE_TRACK_COLOR = "orange"
CONTOUR_GALLERY_TRACK_COLORWAY = "jet_r"
CONTOUR_GALLERY_CONTOUR_COLORWAY = "BuGn"
CONTOUR_GALLERY_CONTOUR_LINE_STYLE = "--"

if CONTOUR_GALLERY_PRESELECT_TRACKS:
    gallery_tracks_df, gallery_track_selection_df = contours.select_tracks_for_gallery_cells(
        visual_instant_df,
        frame_range=CONTOUR_GALLERY_FRAME_RANGE,
        selection_method=CONTOUR_GALLERY_TRACK_SELECTION_METHOD,
        target_tracks_per_cell=CONTOUR_GALLERY_TARGET_TRACKS_PER_CELL,
        min_track_points=CONTOUR_GALLERY_TRACK_MIN_POINTS,
        min_track_length_frames=CONTOUR_GALLERY_TRACK_MIN_LENGTH_FRAMES,
        population_col=CONTOUR_GALLERY_POPULATION_COL,
        populations=CONTOUR_GALLERY_POPULATIONS,
        tracks_per_population=CONTOUR_GALLERY_TRACKS_PER_POPULATION,
        random_state=42,
        group_col=CELL_KEY_COL,
        track_id_col=TRACK_ID_COL,
    )
else:
    gallery_tracks_df = visual_instant_df
    gallery_track_selection_df = None

if RUN_CONTOUR_GALLERY:
    contour_gallery = contours.plot_segmented_contour_gallery(
        gallery_tracks_df,
        visual_contours_ok_df,
        n_cells=CONTOUR_GALLERY_N_CELLS,
        selection_method=CONTOUR_GALLERY_SELECTION,
        group_col=CELL_KEY_COL,
        track_id_col=TRACK_ID_COL,
        contour_id_col="contour_id",
        coordinate_units=CONTOUR_COORDINATE_UNITS,
        x_col=CONTOUR_X_COL,
        y_col=CONTOUR_Y_COL,
        n_cols=4,
        figsize_per_cell=(2, 2),
        contour_colorway=CONTOUR_GALLERY_CONTOUR_COLORWAY,
        contour_line_width=1.0,
        contour_line_style=CONTOUR_GALLERY_CONTOUR_LINE_STYLE,
        show_contour_ids=True,
        show_panel_numbers=True,
        panel_number_start=1,
        show_tracks=CONTOUR_GALLERY_SHOW_TRACKS,
        track_color_mode=CONTOUR_GALLERY_TRACK_COLOR_MODE,
        track_color=CONTOUR_GALLERY_SINGLE_TRACK_COLOR,
        track_colorway=CONTOUR_GALLERY_TRACK_COLORWAY,
        track_line_width=0.7,
        track_alpha=0.85,
        max_tracks_per_cell=None if CONTOUR_GALLERY_PRESELECT_TRACKS else CONTOUR_GALLERY_TARGET_TRACKS_PER_CELL,
        frame_range=CONTOUR_GALLERY_FRAME_RANGE,
        show_raw_projection=CONTOUR_GALLERY_SHOW_RAW_PROJECTION,
        raw_projection=CONTOUR_GALLERY_RAW_PROJECTION,
        raw_projection_frame_range=CONTOUR_GALLERY_RAW_FRAME_RANGE,
        raw_max_frames=CONTOUR_GALLERY_RAW_MAX_FRAMES,
        raw_master_dir=RAW_MASTER_DIR,
        raw_contrast_limits=CONTOUR_GALLERY_RAW_CONTRAST_LIMITS,
        invert_raw=CONTOUR_GALLERY_INVERT_RAW,
        raw_cmap=CONTOUR_GALLERY_RAW_CMAP,
        raw_lut=CONTOUR_GALLERY_RAW_LUT,
        save_path=None,
        dpi=150,
        show_plot=True,
    )
    gallery_lookup_df = contour_gallery["cell_lookup_df"]
    print("\nContour gallery cell lookup:")
    for _, lookup_row in gallery_lookup_df.iterrows():
        print(
            f"{lookup_row.get('gallery_number', '?')}: "
            f"n_contours={lookup_row.get('n_contours', '')} | "
            f"foldername={lookup_row.get('foldername', '')} | "
            f"filename={lookup_row.get('filename', lookup_row.get('cell', ''))}"
        )
    display(gallery_lookup_df)

## 5. Single-Cell Movie And Static Selection

All single-cell display settings live in this one cell. Run this before QC, movie preview/export, or static PNG export.

### Single-Cell Display Parameter Guide

`SELECT_CELL_MODE`: how to choose the cell for movie/stills. Use `gallery_number` to pick from the gallery, `manual` to use `MANUAL_FILENAME`, `random`, `most_tracks`, or `most_tracks_in_population`.

`GALLERY_NUMBER`: the numbered gallery panel to use when `SELECT_CELL_MODE = "gallery_number"`.

`MANUAL_FILENAME`: exact filename to use when `SELECT_CELL_MODE = "manual"`.

`SELECT_FINAL_POPULATION`: population used only when selecting `most_tracks_in_population`.

`RANDOM_SEED`: makes random cell/track choices reproducible.

`FRAME_RANGE_MODE`: how to choose the time window. Use `all`, `manual`, `track_rich`, or `population_rich`.

`MANUAL_FRAME_RANGE`: explicit `(start_frame, end_frame)` used only when `FRAME_RANGE_MODE = "manual"`.

`TARGET_TRACKS`: target number of tracks for track-rich windows and non-balanced track selection.

`WINDOW_SECONDS`: length of the automatically selected movie/still time window.

`START_FRAME`: optional earliest frame to start searching from. Leave `None` to search from the beginning.

`RANGE_MIN_TRACK_POINTS`: minimum points a track needs inside a candidate window to count toward frame-range selection.

`POPULATION_COL_FOR_MOVIE`: column containing population labels, usually `final_population`.

`POPULATIONS_FOR_MOVIE`: populations required for `population_rich` windows and balanced track selection.

`MIN_TRACKS_PER_POPULATION_IN_WINDOW`: minimum tracks per listed population required in a `population_rich` window.

`TRACK_SELECTION_METHOD`: how to choose displayed tracks after the frame range is selected. Use `longest`, `most_points`, `random`, or `balanced_by_population`.

`TRACKS_PER_POPULATION`: number of tracks to show per population when using `balanced_by_population`.

`TRACK_MIN_POINTS`: minimum points a displayed track must have in the selected frame range.

`TRACK_MIN_LENGTH_FRAMES`: optional minimum duration in frames for displayed tracks. Leave `None` to ignore.

`TRACK_RANDOM_STATE`: makes random track selection reproducible.

In [ ]:
# =============================================================================
# ALL SINGLE-CELL MOVIE/STATIC DISPLAY SELECTION SETTINGS
# =============================================================================

# Cell selection: "gallery_number", "manual", "random", "most_tracks", "most_tracks_in_population"
SELECT_CELL_MODE = "gallery_number"  # how to pick the cell/movie for movie and stills
GALLERY_NUMBER = 7  # gallery panel number to use when SELECT_CELL_MODE is "gallery_number"
MANUAL_FILENAME = None  # exact filename to use when SELECT_CELL_MODE is "manual"
SELECT_FINAL_POPULATION = "bound"  # population used for "most_tracks_in_population"
RANDOM_SEED = 42  # reproducible random cell/track choices

# Frame range: "all", "manual", "track_rich", "population_rich"
FRAME_RANGE_MODE = "population_rich"  # how to pick the time window
MANUAL_FRAME_RANGE = None  # explicit (start_frame, end_frame) for manual mode; e.g. (100, 1600)
TARGET_TRACKS = 100  # target number of tracks for auto window/track selection
WINDOW_SECONDS = 15  # duration of auto-selected window
START_FRAME = None  # optional earliest frame for auto search
RANGE_MIN_TRACK_POINTS = 10  # minimum points in-window for a track to count during window search

# Population-aware frame/track selection.
POPULATION_COL_FOR_MOVIE = "final_population"  # column with track population labels
POPULATIONS_FOR_MOVIE = ["bound", "confined", "exploratory", "transport"]  # populations to require/balance
MIN_TRACKS_PER_POPULATION_IN_WINDOW = 1  # minimum tracks per population for population_rich windows

# Track selection: "longest", "most_points", "random", "balanced_by_population"
TRACK_SELECTION_METHOD = "balanced_by_population" if FRAME_RANGE_MODE == "population_rich" else "longest"  # track subset strategy
TRACKS_PER_POPULATION = 3  # tracks per population for balanced_by_population
TRACK_MIN_POINTS = 120  # minimum points in selected window for displayed tracks
TRACK_MIN_LENGTH_FRAMES = None  # optional minimum track duration in frames
TRACK_RANDOM_STATE = 42  # reproducible random track selection

# Execute selection.
# Internally, selection uses cell_key = foldername + filename to avoid mixing repeated filenames.
selected_cell_key, single_cell_df = contours.choose_single_cell_for_display(
    visual_instant_df,
    mode=SELECT_CELL_MODE,
    gallery_lookup_df=gallery_lookup_df if "gallery_lookup_df" in globals() else None,
    gallery_number=GALLERY_NUMBER,
    manual_filename=MANUAL_FILENAME,
    population_col=POPULATION_COL_FOR_MOVIE,
    target_population=SELECT_FINAL_POPULATION,
    random_seed=RANDOM_SEED,
    group_col=CELL_KEY_COL,
    track_id_col=TRACK_ID_COL,
)

selected_filename = single_cell_df.select("filename").unique().item()
selected_foldername = single_cell_df.select("foldername").unique().item()
single_cell_contours_df = visual_contours_ok_df.filter(pl.col(CELL_KEY_COL) == selected_cell_key)

FRAME_RANGE, frame_range_info = contours.choose_frame_range_for_display(
    single_cell_df,
    mode=FRAME_RANGE_MODE,
    manual_frame_range=MANUAL_FRAME_RANGE,
    target_tracks=TARGET_TRACKS,
    window_seconds=WINDOW_SECONDS,
    frame_interval=FRAME_INTERVAL,
    start_frame=START_FRAME,
    min_track_points=RANGE_MIN_TRACK_POINTS,
    population_col=POPULATION_COL_FOR_MOVIE,
    populations=POPULATIONS_FOR_MOVIE,
    min_tracks_per_population=MIN_TRACKS_PER_POPULATION_IN_WINDOW,
    track_id_col=TRACK_ID_COL,
)

track_population_col = POPULATION_COL_FOR_MOVIE if TRACK_SELECTION_METHOD == "balanced_by_population" else None
track_populations = POPULATIONS_FOR_MOVIE if TRACK_SELECTION_METHOD == "balanced_by_population" else None
track_selection = contours.filter_tracks_for_raw_overlay(
    single_cell_df,
    frame_range=FRAME_RANGE,
    target_tracks=TARGET_TRACKS,
    selection_method=TRACK_SELECTION_METHOD,
    min_track_points=TRACK_MIN_POINTS,
    min_track_length_frames=TRACK_MIN_LENGTH_FRAMES,
    population_col=track_population_col,
    populations=track_populations,
    tracks_per_population=TRACKS_PER_POPULATION,
    random_state=TRACK_RANDOM_STATE,
    track_id_col=TRACK_ID_COL,
)

tracks_to_plot_df = track_selection["tracks_df"]
selected_track_df = tracks_to_plot_df
EXPORT_FRAME_RANGE = FRAME_RANGE

print("Selected cell_key:", selected_cell_key)
print("Selected foldername:", selected_foldername)
print("Selected filename:", selected_filename)
print("Rows:", single_cell_df.shape[0])
print("Tracks overall:", single_cell_df.select(pl.col(TRACK_ID_COL).n_unique()).item())
print("Frame range:", FRAME_RANGE)
print("Time range:", tuple(round(f * FRAME_INTERVAL, 2) for f in FRAME_RANGE), "s")
print("Frame range info:", frame_range_info)
print("Selected tracks:", track_selection["n_tracks"])
print("First selected track IDs:", track_selection["selected_track_ids"][:10])

if POPULATION_COL_FOR_MOVIE in single_cell_df.columns:
    display(single_cell_df.group_by(POPULATION_COL_FOR_MOVIE).agg(pl.col(TRACK_ID_COL).n_unique().alias("n_tracks")).sort("n_tracks", descending=True))
    display(tracks_to_plot_df.filter((pl.col("frame") >= FRAME_RANGE[0]) & (pl.col("frame") <= FRAME_RANGE[1])).group_by(POPULATION_COL_FOR_MOVIE).agg(pl.col(TRACK_ID_COL).n_unique().alias("n_tracks")).sort("n_tracks", descending=True))

display(single_cell_contours_df.select(["filename", "contour_id", "area", "circumference", "n_tracks", "n_source_points"]))

## 6. Quick QC Plot

In [ ]:
qc_fig, qc_ax = plt.subplots(figsize=(5, 5), dpi=180)
selected_track_pdf = selected_track_df.to_pandas()
contour = contours.get_contour_array(single_cell_contours_df, coordinate_units="pixels")

# Match the raw-overlay orientation selected by the alignment diagnostic.
qc_width = int(selected_track_pdf[CONTOUR_X_COL].max() + 1)
qc_height = int(selected_track_pdf[CONTOUR_Y_COL].max() + 1)
if "contour_x" in single_cell_contours_df.columns:
    qc_width = max(qc_width, int(np.nanmax(contour[:, 0]) + 1))
    qc_height = max(qc_height, int(np.nanmax(contour[:, 1]) + 1))
qc_width = max(qc_width, 150)
qc_height = max(qc_height, 150)

def _apply_display_transform(x, y, width=qc_width, height=qc_height, transform=RAW_TRACK_SPATIAL_TRANSFORM):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if transform == "flip_x":
        return width - 1 - x, y
    if transform == "flip_y":
        return x, height - 1 - y
    if transform == "flip_xy":
        return width - 1 - x, height - 1 - y
    return x, y

for _, track in selected_track_pdf.sort_values("frame").groupby(TRACK_ID_COL):
    if len(track) >= 2:
        tx, ty = _apply_display_transform(track[CONTOUR_X_COL], track[CONTOUR_Y_COL])
        qc_ax.plot(tx, ty, lw=0.8, alpha=0.75)

contour_x, contour_y = _apply_display_transform(contour[:, 0], contour[:, 1])
qc_ax.plot(contour_x, contour_y, color="tab:red", lw=2.0, ls="--", label="saved minimal bridge contour")
qc_ax.set_title(f"Selected tracks and saved contour\n{selected_filename}\ntransform={RAW_TRACK_SPATIAL_TRANSFORM}", fontsize=8)
qc_ax.set_aspect("equal", adjustable="box")
qc_ax.invert_yaxis()
qc_ax.axis("off")
qc_ax.legend(loc="upper right", fontsize=7)
plt.show()

## 8. Static Timepoint PNGs

In [ ]:
RUN_STILLS_EXPORT = True
# STILLS_OUTPUT_DIR = Path(
#     f"{RESULTS_DIR}/stills/{Path(str(selected_filename)).stem}_{STILLS_OUTPUT_FORMAT}"
# )

STILLS_STEP_FRAMES = 30  # None = every frame
STILLS_OUTPUT_FORMAT = "svg"  # "png" or "svg"; SVG embeds the raw image as raster but keeps overlays vector
STILLS_RAW_MODE = "final_frame"  # "final_frame" or "sum_window"
STILLS_TRACK_HISTORY = "cumulative"  # "cumulative" or "window"
STILLS_RAW_CMAP = RAW_CMAP
STILLS_RAW_LUT = None  # set to chosen_LUT or JDM_LUTS["name"] to use a Fiji LUT
STILLS_INVERT_RAW = True
STILLS_CONTRAST_LIMITS = RAW_CONTRAST_LIMITS

STILLS_OUTPUT_DIR = Path(
    f"{RESULTS_DIR}/stills/{Path(str(selected_filename)).stem}_{STILLS_OUTPUT_FORMAT}_{STILLS_RAW_CMAP}_lut"
)

# Track color controls.
COLORBLIND_HEX = [
    "#0173B2",  # Blue
    "#DE8F05",  # Orange
    "#029E73",  # Green
    "#CC78BC",  # Purple
    "#CA9161",  # Brown
    "#56B4E9",  # Sky Blue
    "#ECE133",  # Yellow
    "#D55E00",  # Vermillion
]
# STILLS_TRACK_COLOR_MODE = "single"  # "single", "by_track", or "by_column"
# STILLS_SINGLE_TRACK_COLOR = COLORBLIND_HEX[2]
STILLS_TRACK_COLOR_MODE = "by_track"  # "single", "by_track", or "by_column"
STILLS_SINGLE_TRACK_COLOR = 'jet_r'
STILLS_TRACK_COLOR_BY = "final_population"  # used only for "by_column"
STILLS_TRACK_COLORWAY = COLORBLIND_HEX  # can be a matplotlib cmap name or a list of hex colors

# Alignment corrections from the diagnostic cell.
STILLS_SPATIAL_TRANSFORM = None #'flip_xy'  # diagnostic selected flip_xy
STILLS_RAW_FRAME_OFFSET = 0

STILLS_SHOW_TIME_LABEL = True
STILLS_SHOW_SCALE_BAR = True
STILLS_SCALE_BAR_LENGTH_UM = 2.0
STILLS_SCALE_BAR_POSITION = "lower_right"
STILLS_SCALE_BAR_MARGIN_PX = 12
STILLS_SCALE_BAR_LINE_WIDTH = 3
STILLS_SCALE_BAR_COLOR = "k"
STILLS_SCALE_BAR_LABEL = None
STILLS_SCALE_BAR_FONTSIZE = 8

if RUN_STILLS_EXPORT:
    STILLS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    still_result = contours.save_raw_track_timelapse_stills(
        single_cell_df,
        selected_track_df,
        master_dir=RAW_MASTER_DIR,
        contour_df=single_cell_contours_df,
        output_dir=STILLS_OUTPUT_DIR,
        frame_range=EXPORT_FRAME_RANGE,
        still_step_frames=STILLS_STEP_FRAMES,
        output_format=STILLS_OUTPUT_FORMAT,
        raw_mode=STILLS_RAW_MODE,
        track_history=STILLS_TRACK_HISTORY,
        x_col=CONTOUR_X_COL,
        y_col=CONTOUR_Y_COL,
        track_color_mode=STILLS_TRACK_COLOR_MODE,
        track_color=STILLS_SINGLE_TRACK_COLOR,
        color_by=STILLS_TRACK_COLOR_BY,
        colorway=STILLS_TRACK_COLORWAY,
        contour_color='k',
        raw_cmap=STILLS_RAW_CMAP,
        raw_lut=STILLS_RAW_LUT,
        invert_image=STILLS_INVERT_RAW,
        contrast_limits=STILLS_CONTRAST_LIMITS,
        spatial_transform=STILLS_SPATIAL_TRANSFORM,
        raw_frame_offset=STILLS_RAW_FRAME_OFFSET,
        show_time_label=STILLS_SHOW_TIME_LABEL,
        frame_interval=FRAME_INTERVAL,
        show_scale_bar=STILLS_SHOW_SCALE_BAR,
        scale_bar_length_um=STILLS_SCALE_BAR_LENGTH_UM,
        pixel_size_um=PIXEL_SIZE_UM,
        scale_bar_position=STILLS_SCALE_BAR_POSITION,
        scale_bar_margin_px=STILLS_SCALE_BAR_MARGIN_PX,
        scale_bar_line_width=STILLS_SCALE_BAR_LINE_WIDTH,
        scale_bar_color=STILLS_SCALE_BAR_COLOR,
        scale_bar_label=STILLS_SCALE_BAR_LABEL,
        scale_bar_fontsize=STILLS_SCALE_BAR_FONTSIZE,
    )
    print(f"Saved {still_result['n_stills']} stills to {STILLS_OUTPUT_DIR}")

In [ ]:
# Final selected cell lookup.
print("Selected cell_key:")
print(selected_cell_key)
print("\nSelected foldername:")
print(selected_foldername)
print("\nSelected filename:")
print(selected_filename)

## 9. Follow A Single Molecule

This renders per-track crops downstream of the selected cell/tracks above. Use `FOLLOW_VIEW_MODE = "follow"` to keep the molecule centered frame-by-frame, or `"center"` to use one fixed crop centered on the selected track's centroid.

In [ ]:
RUN_FOLLOW_TRACK_EXPORT = True

FOLLOW_OUTPUT_DIR = RESULTS_DIR / "follow_tracks" / Path(str(selected_filename)).stem

# Output toggles: use ("png",), ("napari",), or ("png", "napari").
FOLLOW_RENDER_OUTPUTS = ("png",)
FOLLOW_VIEW_MODE = "follow"  # "follow" or "center"
FOLLOW_OUTPUT_FORMAT = "png"  # "png" or "svg" for still frames

# Track selection: "specified", "random", "population", or "all".
FOLLOW_TRACK_SELECTION_MODE = "random"
FOLLOW_TRACK_IDS = None  # e.g. [12345] when FOLLOW_TRACK_SELECTION_MODE = "specified"
FOLLOW_N_TRACKS = 5
FOLLOW_POPULATION_COL = "final_population"
FOLLOW_POPULATIONS = ["transport"]
FOLLOW_RANDOM_STATE = 42

# Crop and time controls.
FOLLOW_FRAME_RANGE = EXPORT_FRAME_RANGE
FOLLOW_FRAME_STEP = 1
FOLLOW_CROP_SIZE_PX = None  # None = auto from track extent; or int; or (width, height)
FOLLOW_MIN_CROP_SIZE_PX = 64
FOLLOW_CROP_PADDING_PX = 12
FOLLOW_RAW_MODE = "final_frame"  # "final_frame" or "sum_window"
FOLLOW_TRACK_HISTORY = "cumulative"  # "cumulative" or "window"
FOLLOW_HISTORY_LENGTH_FRAMES = None  # optional tail length for PNG history

# Display controls.
FOLLOW_RAW_CMAP = STILLS_RAW_CMAP
FOLLOW_RAW_LUT = STILLS_RAW_LUT
FOLLOW_INVERT_RAW = STILLS_INVERT_RAW
FOLLOW_CONTRAST_LIMITS = STILLS_CONTRAST_LIMITS
FOLLOW_SPATIAL_TRANSFORM = STILLS_SPATIAL_TRANSFORM
FOLLOW_RAW_FRAME_OFFSET = STILLS_RAW_FRAME_OFFSET
FOLLOW_TRACK_COLOR = COLORBLIND_HEX[0]  # must be a real color, not a colormap name
FOLLOW_HEAD_COLOR = COLORBLIND_HEX[5]
FOLLOW_TRACK_LINE_WIDTH = 1.0
FOLLOW_HEAD_SIZE = 12
FOLLOW_DPI = 200
FOLLOW_FPS = 20

# Annotation controls.
FOLLOW_SHOW_TIME_LABEL = True
FOLLOW_SHOW_SCALE_BAR = True
FOLLOW_SCALE_BAR_LENGTH_UM = STILLS_SCALE_BAR_LENGTH_UM
FOLLOW_SCALE_BAR_POSITION = STILLS_SCALE_BAR_POSITION
FOLLOW_SCALE_BAR_MARGIN_PX = STILLS_SCALE_BAR_MARGIN_PX
FOLLOW_SCALE_BAR_LINE_WIDTH = STILLS_SCALE_BAR_LINE_WIDTH
FOLLOW_SCALE_BAR_COLOR = STILLS_SCALE_BAR_COLOR
FOLLOW_SCALE_BAR_LABEL = STILLS_SCALE_BAR_LABEL
FOLLOW_SCALE_BAR_FONTSIZE = STILLS_SCALE_BAR_FONTSIZE

if RUN_FOLLOW_TRACK_EXPORT:
    follow_result = contours.render_follow_track_views(
        single_cell_df,
        selected_track_df,
        output_dir=FOLLOW_OUTPUT_DIR,
        frame_range=FOLLOW_FRAME_RANGE,
        master_dir=RAW_MASTER_DIR,
        render_outputs=FOLLOW_RENDER_OUTPUTS,
        view_mode=FOLLOW_VIEW_MODE,
        track_selection_mode=FOLLOW_TRACK_SELECTION_MODE,
        track_ids=FOLLOW_TRACK_IDS,
        population_col=FOLLOW_POPULATION_COL,
        populations=FOLLOW_POPULATIONS,
        n_tracks=FOLLOW_N_TRACKS,
        random_state=FOLLOW_RANDOM_STATE,
        track_id_col=TRACK_ID_COL,
        x_col=CONTOUR_X_COL,
        y_col=CONTOUR_Y_COL,
        spatial_transform=FOLLOW_SPATIAL_TRANSFORM,
        raw_frame_offset=FOLLOW_RAW_FRAME_OFFSET,
        crop_size_px=FOLLOW_CROP_SIZE_PX,
        min_crop_size_px=FOLLOW_MIN_CROP_SIZE_PX,
        crop_padding_px=FOLLOW_CROP_PADDING_PX,
        frame_step=FOLLOW_FRAME_STEP,
        raw_mode=FOLLOW_RAW_MODE,
        output_format=FOLLOW_OUTPUT_FORMAT,
        dpi=FOLLOW_DPI,
        fps=FOLLOW_FPS,
        raw_cmap=FOLLOW_RAW_CMAP,
        raw_lut=FOLLOW_RAW_LUT,
        contrast_limits=FOLLOW_CONTRAST_LIMITS,
        invert_raw=FOLLOW_INVERT_RAW,
        track_history=FOLLOW_TRACK_HISTORY,
        history_length_frames=FOLLOW_HISTORY_LENGTH_FRAMES,
        track_color=FOLLOW_TRACK_COLOR,
        track_line_width=FOLLOW_TRACK_LINE_WIDTH,
        head_color=FOLLOW_HEAD_COLOR,
        head_size=FOLLOW_HEAD_SIZE,
        show_time_label=FOLLOW_SHOW_TIME_LABEL,
        frame_interval=FRAME_INTERVAL,
        show_scale_bar=FOLLOW_SHOW_SCALE_BAR,
        scale_bar_length_um=FOLLOW_SCALE_BAR_LENGTH_UM,
        pixel_size_um=PIXEL_SIZE_UM,
        scale_bar_position=FOLLOW_SCALE_BAR_POSITION,
        scale_bar_margin_px=FOLLOW_SCALE_BAR_MARGIN_PX,
        scale_bar_line_width=FOLLOW_SCALE_BAR_LINE_WIDTH,
        scale_bar_color=FOLLOW_SCALE_BAR_COLOR,
        scale_bar_label=FOLLOW_SCALE_BAR_LABEL,
        scale_bar_fontsize=FOLLOW_SCALE_BAR_FONTSIZE,
    )
    print("Selected follow track IDs:", follow_result["selected_track_ids"])
    print("Raw movie path:", follow_result["raw_movie_path"])
    for record in follow_result["outputs"]:
        print(record)
else:
    print("Follow-track export skipped. Set RUN_FOLLOW_TRACK_EXPORT = True to run.")

# Part B — live state + softmax movies

# Live state-switching + softmax movie — raw-motion transformer 2026

One track, two synchronized panels:

- **Left:** the track in XY, drawn as it moves, coloured by `transformer_behavior_state`.
  State changes trigger a head pulse + gold trail glow (same idea as
  `sptviz.render_state_change_track_movie`).
- **Right:** the smooth stacked softmax area (Fig 1e), revealed up to the same instant,
  with a thin assignment ribbon underneath.

Both panels run on **one continuous clock** (sub-frame interpolation), so the head of the
track and the leading edge of the stacked area move smoothly together instead of jumping
window-by-window.

**How to use**

1. Edit the **dataset filter** cell (first code cell) — which molecule / genotype / cell type / file.
2. Edit the **track picker** knobs (same cell) — mixed vs a named state, length, seed.
3. Run all. The parquet is cached in memory, so re-running after changing the filter or the
   seed is fast.

Timing: each window's softmax is anchored at its **window centre** (`SOFTMAX_ANCHOR`), and each
frame takes the assignment of the containing window with the nearest centre
(`FRAME_STATE_POLICY = "nearest_center"`). With these defaults the track colour switches at the
same moment the stacked area crosses over.

In [ ]:
# =============================================================================
# 1. DATASET FILTER — which tracks are eligible
# =============================================================================
# Keys are columns of the windowed parquet (all are constant per track).
#   None                 -> no filter on this column
#   "HTT"                -> keep rows equal to this value
#   ["HTT", "kinesin"]   -> keep rows whose value is in this list
# The load cell prints every available value per column.
DATASET_FILTER = {
    "mol": ["HTT"],                     # HTT | myosin | kinesin | dynein
    "geno": None,                       # 20H20S | 150H20S | 20H150S | 77H20S | RUES2 | HTTKO | ...
    "type": None,                       # vent | cort | ES
    "location": None,                   # CB | GC | NN | ES
    "cell_type": None,                  # vent | cort | ES | neuron
    "cell": None,                       # neuron | ES
    "group": None,                      # HTT150 | HTT77 | HTT56 | wtkomt | noco | hap40 | endoH
    "condition": None,
    "nocodazole_concentration": None,   # none | 0ngml | 50ngml | 1000ngml
    "replicate": None,                  # none | R3
}
FILENAME_CONTAINS = None    # substring or list of substrings, e.g. "20H-20S_001"
EXTRA_FILTER = None         # any polars expression, e.g. pl.col("n_valid_frames") >= 60
DROP_UNGATED = True

# =============================================================================
# 2. TRACK PICKER — which track inside the filtered dataset
# =============================================================================
PICK_CONTENT = "mixed"              # "mixed" | "bound" | "confined" | "transport" | "exploratory" | "any"
PICK_REQUIRE_CONTINUOUS = True      # no missing windows inside the track
PICK_EXCLUDE_UNRESOLVED = True      # drop tracks with any unresolved_* window
PICK_MIN_WINDOWS = 8
PICK_MAX_WINDOWS = 40               # None = no maximum (long tracks -> long movies)
PICK_MIN_TARGET_FRACTION = 0.35     # named state: minimum fraction of windows
PICK_MIN_DISTINCT_FOR_MIXED = 3     # mixed: at least this many distinct assignments
PICK_MIN_SWITCHES_FOR_MIXED = 3     # mixed: at least this many assignment changes

PICK_RANDOM_SEED = 7                # <-- change to draw another track
PICK_AMONG_TOP = 12                 # draw one track from this many top-scoring candidates
FORCE_UNIQUE_ID = None              # set to a unique_id string to skip the picker

In [ ]:
# =============================================================================
# 3. MOVIE KNOBS
# =============================================================================
MOVIE_SPEED = 0.35          # playback speed vs real time (1.0 = real time; 0.35 = ~3x slow motion)
MOVIE_FPS = 30
MOVIE_DPI = 150
MOVIE_FIGSIZE = (11.2, 4.8) # inches; keep figsize * dpi even for H.264
MOVIE_WRITE_MP4 = True
MOVIE_WRITE_GIF = False     # GIFs are large; MP4 is the smooth one
LEAD_IN_S = 0.4             # movie seconds held on t = 0 before playback
TAIL_HOLD_S = 1.2           # movie seconds held on the last frame

# --- smoothing / timing ---
SOFTMAX_SMOOTH_N = 1500     # dense samples of the PCHIP-interpolated softmax
SOFTMAX_ANCHOR = "center"   # "center" | "start" frame of each window
FRAME_STATE_POLICY = "nearest_center"  # "nearest_center" | "later_window" | "earlier_window"
TRACK_SMOOTH = 1.2          # Gaussian sigma in frames; 0 = off
TRACK_CURVE = 4             # cubic samples per frame step; 1 = straight chords
TRACK_LINEWIDTH = 2.4

# --- XY panel ---
XY_SPAN_UM = None           # None = fit the whole track; number = fixed square box (um)
XY_PAD_FRAC = 0.12
XY_INVERT_Y = False         # True = image orientation (y down)
SCALE_BAR_UM = 0.5
HEAD_SIZE_PT = 7.0

# --- state-change flash ---
FLASH_ON_CHANGE = True
FLASH_DURATION_S = 0.55     # movie seconds the pulse lasts
FREEZE_ON_CHANGE_S = 0.0    # movie seconds the clock pauses at each change (0 = keep flowing)
FLASH_GOLD = (1.0, 0.84, 0.15)
HEAD_PULSE_SCALE = 1.0

# --- softmax panel ---
SHOW_ASSIGNMENT_RIBBON = True
SHOW_FUTURE_GHOST = False   # faint full stack behind the reveal
MARK_CHANGES_ON_AREA = True # dotted vertical line at each assignment change

# --- raw-data panel (leftmost subplot when on; off = movies look exactly as before) ---
RAW_PANEL = True
RAW_PANEL_WIDTH_IN = 4.2            # figure width added for the panel (inches)
RAW_ON_MISSING = "skip_panel"       # "skip_panel" | "error" when the raw TIFF can't be found/read
RAW_MASTER_DIR = None               # override experiment folder (default: foldername / source_folder columns)
RAW_PATH_OVERRIDES = {}             # {unique_id: r"path/to/movie.tif"}
# LUT
RAW_CMAP = "gray"                   # any matplotlib colormap name (used when RAW_USE_FIJI_LUT is False)
RAW_USE_FIJI_LUT = False
RAW_FIJI_LUT_DIR = None              # None = fiji_lut_dir from paths.local.yaml
RAW_FIJI_LUT_NAME = "JDM_Ink Wash Yellow"  # file stem in RAW_FIJI_LUT_DIR, or a full .lut path; see list_fiji_luts()
RAW_REVERSE_LUT = False             # flip the colormap direction
RAW_INVERT = False                  # invert intensities (dark spots on a light background)
# contrast
RAW_CONTRAST_LIMITS = None          # (lo, hi) in raw counts; None = use percentiles
RAW_CONTRAST_PERCENTILES = (1.0, 99.7)
RAW_CONTRAST_PER_FRAME = False      # True = re-stretch each frame (can flicker); False = one stretch per movie
RAW_GAMMA = 1.0                     # <1 brightens dim signal, >1 darkens background
RAW_TEMPORAL_AVG = 1                # rolling mean over N raw frames (1 = off)
RAW_FRAME_BLEND = False             # crossfade between consecutive raw frames
RAW_INTERPOLATION = "nearest"       # "nearest" | "bilinear" | "bicubic"
# field of view
RAW_FOV_MODE = "track_crop"         # "track_crop" (fixed box) | "follow" (box moves with the particle)
RAW_SPAN_UM = "match_xy"            # "match_xy" (same box as XY panel) | number (um) | None = fit track + RAW_PAD_UM
RAW_PAD_UM = 0.5
RAW_FOLLOW_BOX_UM = 3.0             # box size in "follow" mode
RAW_FORCE_XY_IMAGE_ORIENTATION = True  # flip the XY panel to y-down so it matches the raw image
# track overlay
RAW_SHOW_TRACK = True
RAW_TRACK_LINEWIDTH = 1.6
RAW_TRACK_ALPHA = 0.85
RAW_TRAIL_MODE = "cumulative"       # "cumulative" | "tail"
RAW_TAIL_S = 0.5                    # track seconds of trail in "tail" mode
RAW_TAIL_FADE = True                # older part of the tail fades out
RAW_HEAD_STYLE = "ring"             # "ring" (hollow, spot stays visible) | "dot" | "none"
RAW_HEAD_SIZE_PT = 10.0
RAW_FLASH_GLOW = True               # gold glow on the overlay at state changes
RAW_FLASH_WHITEN = 0.0              # 0-1: brief white lift of the image at state changes
RAW_SCALE_BAR_UM = 1.0
RAW_TEXT_COLOR = "auto"             # "auto" = white on dark images, black when inverted
RAW_SHOW_TIMER = False

In [ ]:
# =============================================================================
# Imports, paths, palettes
# =============================================================================
import sys
import time
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import Image, Video, display
from matplotlib import animation
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

sys.path.insert(0, "..")
from sptnano import config

paths = config.load_paths()

%matplotlib inline

try:
    import imageio_ffmpeg
    mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except Exception as exc:
    print("imageio_ffmpeg not available; MP4 needs ffmpeg on PATH:", exc)

MODEL_VARIANT = "block3"
EVAL_DIR = paths["evaluation_dir"]
WINDOWED_PARQUET = EVAL_DIR / "heldout_test_windows_with_transformer_behavior_state.parquet"
INSTANT_PARQUET = EVAL_DIR / "heldout_test_instant_with_transformer_behavior_state.parquet"

PLOTS_DIR = paths["plots_dir"]
MOVIE_DIR = PLOTS_DIR / "nature_methods_transformer_2026" / "live_state_softmax_movies" / MODEL_VARIANT
MOVIE_DIR.mkdir(parents=True, exist_ok=True)

FRAME_INTERVAL = config.TIME_BETWEEN_FRAMES
PIXEL_SIZE_UM = config.PIXELSIZE_MICRONS
TIME_WINDOW = config.TIME_WINDOW
OVERLAP = config.OVERLAP
STATE_COL = "transformer_behavior_state"

PURE_STATES = ["bound", "confined", "transport", "exploratory"]
BEHAVIOR_STATE_PALETTE = {
    "bound": "#1b9e77",
    "confined": "#d95f02",
    "transport": "#7570b3",
    "exploratory": "#e7298a",
}


def _hex_to_rgb(h):
    h = str(h).lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def _mix_hex(a, b, t=0.5):
    ra, ga, ba = _hex_to_rgb(a)
    rb, gb, bb = _hex_to_rgb(b)
    mix = (ra + (rb - ra) * t, ga + (gb - ga) * t, ba + (bb - ba) * t)
    return "#{:02x}{:02x}{:02x}".format(*(max(0, min(255, int(v))) for v in mix))


def mixture_state_color(label):
    s = str(label)
    if s in BEHAVIOR_STATE_PALETTE:
        return BEHAVIOR_STATE_PALETTE[s]
    if s.startswith("unresolved_"):
        base = s[len("unresolved_"):]
        if base in BEHAVIOR_STATE_PALETTE:
            return _mix_hex(BEHAVIOR_STATE_PALETTE[base], "#d0d0d0", 0.45)
        return "#b0b0b0"
    parts = s.split("_")
    if len(parts) == 2 and all(p in BEHAVIOR_STATE_PALETTE for p in parts):
        return _mix_hex(BEHAVIOR_STATE_PALETTE[parts[0]], BEHAVIOR_STATE_PALETTE[parts[1]], 0.5)
    return "#9a9a9a"


def mixture_state_order(labels):
    seen = list(dict.fromkeys(str(s) for s in labels))
    pures = [s for s in PURE_STATES if s in seen]
    unresolved = sorted(s for s in seen if s.startswith("unresolved_"))
    pairs = sorted(s for s in seen if s not in pures and s not in unresolved)
    return pures + pairs + unresolved


print("Variant:", MODEL_VARIANT)
print("Windowed:", WINDOWED_PARQUET)
print("Instant: ", INSTANT_PARQUET)
print("Movies:  ", MOVIE_DIR)

if RAW_FIJI_LUT_DIR is None:
    RAW_FIJI_LUT_DIR = str(paths["fiji_lut_dir"])

## Load the full windowed table (cached)

Loads every non-embedding column once and keeps it as `windowed_full` (polars). Re-running the
notebook reuses it unless the parquet path changes. The printout lists the values you can put in
`DATASET_FILTER`, with **track** counts.

In [ ]:
if globals().get("_WINDOWED_CACHE_PATH") != str(WINDOWED_PARQUET):
    _names = pl.scan_parquet(WINDOWED_PARQUET).collect_schema().names()
    windowed_full = pl.read_parquet(WINDOWED_PARQUET, columns=[c for c in _names if not c.startswith("emb_")])
    if "window_uid" in windowed_full.columns:
        windowed_full = windowed_full.unique(subset=["window_uid"], keep="first", maintain_order=True)
    _WINDOWED_CACHE_PATH = str(WINDOWED_PARQUET)
    print("Loaded windowed_full:", windowed_full.shape)
else:
    print("Using cached windowed_full:", windowed_full.shape)

print(f"{windowed_full['unique_id'].n_unique():,} tracks in the full table\n")
for col in DATASET_FILTER:
    if col not in windowed_full.columns:
        print(f"{col}: <not a column>")
        continue
    vc = (
        windowed_full.group_by(pl.col(col).cast(pl.Utf8))
        .agg(pl.col("unique_id").n_unique().alias("n_tracks"))
        .sort("n_tracks", descending=True)
    )
    shown = ", ".join(f"{r[col]} ({r['n_tracks']:,})" for r in vc.head(20).iter_rows(named=True))
    more = f"  ... +{vc.height - 20} more" if vc.height > 20 else ""
    print(f"{col}: {shown}{more}")

## Apply the dataset filter

In [ ]:
UNGATED = ["ungatedgate1", "ungated", "unknown"]


def _as_list(val):
    if isinstance(val, (str, int, float, bool)):
        return [val]
    return list(val)


def apply_dataset_filter(df, dataset_filter=None, use_extra=True):
    dataset_filter = DATASET_FILTER if dataset_filter is None else dataset_filter
    steps = [("start", df.height, df["unique_id"].n_unique())]
    if DROP_UNGATED:
        for col in [c for c in ("final_population", "final_population_certain", STATE_COL) if c in df.columns]:
            df = df.filter(~pl.col(col).cast(pl.Utf8).is_in(UNGATED))
        steps.append(("drop ungated", df.height, df["unique_id"].n_unique()))
    for col, val in dataset_filter.items():
        if val is None:
            continue
        if col not in df.columns:
            print(f"  [skip] {col!r} is not a column")
            continue
        vals = [str(v) for v in _as_list(val)]
        df = df.filter(pl.col(col).cast(pl.Utf8).is_in(vals))
        steps.append((f"{col} in {vals}", df.height, df["unique_id"].n_unique()))
    if use_extra and FILENAME_CONTAINS:
        pats = [str(p) for p in _as_list(FILENAME_CONTAINS)]
        expr = pl.lit(False)
        for p in pats:
            expr = expr | pl.col("filename").cast(pl.Utf8).str.contains(p, literal=True)
        df = df.filter(expr)
        steps.append((f"filename contains {pats}", df.height, df["unique_id"].n_unique()))
    if use_extra and EXTRA_FILTER is not None:
        df = df.filter(EXTRA_FILTER)
        steps.append(("EXTRA_FILTER", df.height, df["unique_id"].n_unique()))
    report = pd.DataFrame(steps, columns=["step", "n_windows", "n_tracks"])
    return df, report


windowed_sel, filter_report = apply_dataset_filter(windowed_full)
display(filter_report)
if windowed_sel.height == 0:
    raise ValueError("DATASET_FILTER removed every row — relax it.")

summary_cols = [c for c in ("mol", "geno", "type", "location") if c in windowed_sel.columns]
display(
    windowed_sel.group_by(summary_cols)
    .agg(pl.col("unique_id").n_unique().alias("n_tracks"), pl.len().alias("n_windows"))
    .sort("n_tracks", descending=True)
    .to_pandas()
)

## Pick a track

Scores every track in the filtered table (vectorised in polars), applies the picker knobs,
then draws one of the top `PICK_AMONG_TOP` candidates with `PICK_RANDOM_SEED`.

In [ ]:
def build_track_pool(df):
    d = (
        df.select(["unique_id", "filename", "time_window", STATE_COL])
        .with_columns(pl.col(STATE_COL).cast(pl.Utf8), pl.col("unique_id").cast(pl.Utf8))
        .sort(["unique_id", "time_window"])
        .with_columns(
            (pl.col(STATE_COL) != pl.col(STATE_COL).shift(1).over("unique_id"))
            .fill_null(False).cast(pl.Int32).alias("_switch"),
            (pl.col("time_window") - pl.col("time_window").shift(1).over("unique_id")).alias("_gap"),
        )
    )
    agg = d.group_by("unique_id").agg(
        pl.col("filename").first(),
        pl.len().alias("n_windows"),
        pl.col("_gap").max().fill_null(1).alias("max_tw_gap"),
        pl.col("time_window").n_unique().alias("_n_tw"),
        (pl.col("time_window").max() - pl.col("time_window").min() + 1).alias("_tw_span"),
        pl.col(STATE_COL).n_unique().alias("n_distinct"),
        pl.col("_switch").sum().alias("n_switches"),
        pl.col(STATE_COL).str.starts_with("unresolved_").any().alias("has_unresolved"),
        *[(pl.col(STATE_COL) == s).mean().alias(f"frac_{s}") for s in PURE_STATES],
    )
    ent = (
        d.group_by(["unique_id", STATE_COL]).len()
        .with_columns((pl.col("len") / pl.col("len").sum().over("unique_id")).alias("_p"))
        .group_by("unique_id")
        .agg((-(pl.col("_p") * pl.col("_p").log())).sum().alias("assign_entropy"))
    )
    pool = (
        agg.join(ent, on="unique_id", how="left")
        .with_columns(
            ((pl.col("n_windows") == pl.col("_n_tw")) & (pl.col("_n_tw") == pl.col("_tw_span"))).alias("continuous"),
            (pl.col("n_switches") * (1.0 + pl.col("assign_entropy"))).alias("mix_score"),
        )
        .drop(["_n_tw", "_tw_span"])
    )
    return pool.to_pandas()


def filter_track_pool(pool):
    df = pool.copy()
    n0 = len(df)
    if PICK_REQUIRE_CONTINUOUS:
        df = df.loc[df["continuous"]]
    if PICK_EXCLUDE_UNRESOLVED:
        df = df.loc[~df["has_unresolved"]]
    df = df.loc[df["n_windows"] >= int(PICK_MIN_WINDOWS)]
    if PICK_MAX_WINDOWS is not None:
        df = df.loc[df["n_windows"] <= int(PICK_MAX_WINDOWS)]
    content = str(PICK_CONTENT).strip().lower()
    if content in PURE_STATES:
        df = df.loc[df[f"frac_{content}"] >= float(PICK_MIN_TARGET_FRACTION)]
        df = df.sort_values([f"frac_{content}", "n_switches", "n_windows"], ascending=False)
    elif content == "mixed":
        df = df.loc[
            (df["n_distinct"] >= int(PICK_MIN_DISTINCT_FOR_MIXED))
            & (df["n_switches"] >= int(PICK_MIN_SWITCHES_FOR_MIXED))
        ]
        df = df.sort_values(["mix_score", "n_distinct", "n_switches"], ascending=False)
    else:
        df = df.sort_values(["n_windows", "mix_score"], ascending=False)
    print(f"Picker: {n0:,} tracks -> {len(df):,} after continuity/unresolved/length/content={content}")
    return df.reset_index(drop=True)


track_pool = build_track_pool(windowed_sel)
if FORCE_UNIQUE_ID is not None:
    PICK_UID = str(FORCE_UNIQUE_ID)
    if PICK_UID not in set(track_pool["unique_id"]):
        raise KeyError(f"FORCE_UNIQUE_ID={PICK_UID!r} is not in the filtered table")
    print("FORCE_UNIQUE_ID:", PICK_UID)
else:
    candidates = filter_track_pool(track_pool)
    if candidates.empty:
        raise ValueError("No track survived the picker — relax PICK_* knobs or DATASET_FILTER.")
    top = candidates.head(max(1, min(int(PICK_AMONG_TOP), len(candidates))))
    pick_i = int(np.random.default_rng(int(PICK_RANDOM_SEED)).integers(0, len(top)))
    PICK_UID = str(top.iloc[pick_i]["unique_id"])
    print(f"seed={PICK_RANDOM_SEED}  among_top={len(top)}  index={pick_i}  ->  {PICK_UID}")
    display(top.round(3))

seq = (
    windowed_sel.filter(pl.col("unique_id").cast(pl.Utf8) == PICK_UID)
    .sort("time_window")
    .to_pandas()
    .reset_index(drop=True)
)
print(f"\n{PICK_UID}: {len(seq)} windows  file={seq['filename'].iloc[0]}")
print(seq[STATE_COL].astype(str).tolist())

## Build the continuous-time track + softmax

- XY comes from the instant parquet (one row per frame; overlapping windows share frames).
- Per-frame assignment follows `FRAME_STATE_POLICY`.
- The softmax is PCHIP-interpolated through the window anchors, renormalised, and held flat
  before the first / after the last anchor.
- The track is Gaussian-smoothed, cubic-resampled, and every dense point carries its own time,
  so the head can sit *between* frames.

In [ ]:
from scipy.interpolate import PchipInterpolator, interp1d
from scipy.ndimage import gaussian_filter1d


def load_track_instant(uid):
    lf = pl.scan_parquet(INSTANT_PARQUET)
    names = lf.collect_schema().names()
    cols = [c for c in ("unique_id", "window_uid", "frame", "time_s", "x_um", "y_um", "x", "y") if c in names]
    inst = lf.filter(pl.col("unique_id").cast(pl.Utf8) == str(uid)).select(cols).collect().to_pandas()
    if inst.empty:
        raise ValueError(f"No instant rows for {uid!r}")
    if "x_um" not in inst.columns:
        inst["x_um"] = inst["x"].astype(float) * PIXEL_SIZE_UM
        inst["y_um"] = inst["y"].astype(float) * PIXEL_SIZE_UM
    return inst.sort_values("frame").drop_duplicates("frame", keep="last").reset_index(drop=True)


def softmax_matrix(seq):
    P = np.vstack([
        pd.to_numeric(seq[f"prob_{s}"], errors="coerce").to_numpy(dtype=float)
        if f"prob_{s}" in seq.columns else np.zeros(len(seq))
        for s in PURE_STATES
    ])
    P = np.nan_to_num(P, nan=0.0)
    return P / np.where(P.sum(axis=0) > 0, P.sum(axis=0), 1.0)


def window_frames(seq):
    fs = pd.to_numeric(seq["frame_start"], errors="coerce").to_numpy(dtype=float)
    if "frame_end" in seq.columns:
        fe = pd.to_numeric(seq["frame_end"], errors="coerce").to_numpy(dtype=float)
    else:
        fe = fs + TIME_WINDOW - 1
    anchor = fs if str(SOFTMAX_ANCHOR).lower() == "start" else 0.5 * (fs + fe)
    return anchor, fs, fe


def per_frame_states(frames, seq):
    anchor, fs, fe = window_frames(seq)
    labels = seq[STATE_COL].astype(str).to_numpy()
    f = np.asarray(frames, dtype=float)[:, None]
    inside = (f >= fs[None, :]) & (f <= fe[None, :])
    dist = np.abs(f - anchor[None, :])
    nearest_any = dist.argmin(axis=1)
    order = np.broadcast_to(np.arange(len(seq))[None, :], inside.shape)
    policy = str(FRAME_STATE_POLICY).lower()
    if policy == "later_window":
        idx = np.where(inside, order, -1).max(axis=1)
    elif policy == "earlier_window":
        idx = np.where(inside, order, len(seq)).min(axis=1)
        idx = np.where(idx < len(seq), idx, -1)
    else:
        idx = np.where(inside.any(axis=1), np.where(inside, dist, np.inf).argmin(axis=1), -1)
    idx = np.where(idx >= 0, idx, nearest_any)
    return labels[idx]


def build_movie_data(seq, inst):
    frames = inst["frame"].to_numpy(dtype=float)
    f0 = frames[0]
    t_frame = (frames - f0) * FRAME_INTERVAL
    t_end = float(t_frame[-1])
    states = per_frame_states(frames, seq)

    x = inst["x_um"].to_numpy(dtype=float)
    y = inst["y_um"].to_numpy(dtype=float)
    if TRACK_SMOOTH and TRACK_SMOOTH > 0 and len(x) >= 3:
        x = gaussian_filter1d(x, sigma=float(TRACK_SMOOTH), mode="nearest")
        y = gaussian_filter1d(y, sigma=float(TRACK_SMOOTH), mode="nearest")
    n = len(x)
    k = max(1, int(TRACK_CURVE))
    u = np.linspace(0.0, n - 1.0, (n - 1) * k + 1) if n > 1 else np.zeros(1)
    kind = "cubic" if (k > 1 and n >= 4) else "linear"
    idx = np.arange(n, dtype=float)
    if n > 1:
        dx = interp1d(idx, x, kind=kind)(u)
        dy = interp1d(idx, y, kind=kind)(u)
    else:
        dx, dy = x.copy(), y.copy()
    dt = np.interp(u, idx, t_frame)
    seg_state = states[np.clip(np.floor(u[:-1]).astype(int), 0, max(n - 2, 0))] if len(u) > 1 else states[:0]

    anchor, _, _ = window_frames(seq)
    t_anchor = (anchor - f0) * FRAME_INTERVAL
    P = softmax_matrix(seq)
    order = np.argsort(t_anchor)
    t_anchor, P = t_anchor[order], P[:, order]
    t_anchor, uniq = np.unique(t_anchor, return_index=True)
    P = P[:, uniq]
    t_soft = np.linspace(0.0, t_end, int(max(SOFTMAX_SMOOTH_N, 50)))
    tq = np.clip(t_soft, t_anchor.min(), t_anchor.max())
    if len(t_anchor) >= 2:
        P_soft = np.vstack([PchipInterpolator(t_anchor, P[i])(tq) for i in range(P.shape[0])])
    else:
        P_soft = np.repeat(P, len(t_soft), axis=1)
    P_soft = np.clip(P_soft, 0.0, None)
    P_soft /= np.where(P_soft.sum(axis=0) > 0, P_soft.sum(axis=0), 1.0)

    change_i = np.flatnonzero(states[1:] != states[:-1]) + 1
    changes = [(float(t_frame[i]), f"{states[i - 1]} \u2192 {states[i]}") for i in change_i]
    return {
        "t_frame": t_frame, "states": states, "t_end": t_end,
        "dense_x": dx, "dense_y": dy, "dense_t": dt, "seg_state": seg_state,
        "t_anchor": t_anchor, "P_anchor": P, "t_soft": t_soft, "P_soft": P_soft,
        "changes": changes,
    }


def _flash_envelope(x, attack=0.18):
    x = np.clip(x, 0.0, 1.0)
    up = np.sin(0.5 * np.pi * np.clip(x / attack, 0, 1))
    down = np.cos(0.5 * np.pi * np.clip((x - attack) / (1 - attack), 0, 1)) ** 1.5
    return np.where(x < attack, up, down)


def build_timeline(data):
    step = float(MOVIE_SPEED) / float(MOVIE_FPS)
    n_flash = max(1, int(round(FLASH_DURATION_S * MOVIE_FPS)))
    n_freeze = int(round(FREEZE_ON_CHANGE_S * MOVIE_FPS))
    changes = data["changes"] if FLASH_ON_CHANGE else []
    taus, flash_start, flash_label = [], [], []
    for _ in range(int(round(LEAD_IN_S * MOVIE_FPS))):
        taus.append(0.0)
    tau, ci = 0.0, 0
    while tau < data["t_end"] - 1e-12:
        nxt = min(tau + step, data["t_end"])
        while ci < len(changes) and changes[ci][0] <= nxt:
            tc, lab = changes[ci]
            flash_start.append(len(taus))
            flash_label.append(lab)
            for _ in range(n_freeze):
                taus.append(tc)
            ci += 1
        taus.append(nxt)
        tau = nxt
    for _ in range(int(round(TAIL_HOLD_S * MOVIE_FPS))):
        taus.append(data["t_end"])
    taus = np.asarray(taus, dtype=float)
    weight = np.zeros(len(taus))
    label = np.full(len(taus), None, dtype=object)
    for ks, lab in zip(flash_start, flash_label):
        span = np.arange(ks, min(ks + n_flash, len(taus)))
        w = _flash_envelope((span - ks + 0.5) / n_flash)
        take = w > weight[span]
        weight[span[take]] = w[take]
        label[span[take]] = lab
    return taus, weight, label


inst = load_track_instant(PICK_UID)
movie_data = build_movie_data(seq, inst)
taus, flash_w, flash_lab = build_timeline(movie_data)
print(f"instant frames: {len(inst)}   track duration: {movie_data['t_end']:.2f} s")
print(f"state changes: {len(movie_data['changes'])}")
for t_c, lab in movie_data["changes"]:
    print(f"  {t_c:6.2f} s  {lab}")
print(f"movie: {len(taus)} frames at {MOVIE_FPS} fps = {len(taus) / MOVIE_FPS:.1f} s  (speed {MOVIE_SPEED:g}x)")

## Live figure

`LiveStateSoftmaxMovie` builds every artist once; `draw(k)` only moves the clock:
the trail `LineCollection` gets its visible segments plus a partial segment to the
interpolated head, and the stacked area is revealed with a clip rectangle whose right
edge sits exactly at the current time.

In [ ]:
import matplotlib.font_manager as fm
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from scipy.ndimage import uniform_filter1d

_RAW_LUT_CACHE = {}


def _read_fiji_lut(path):
    raw = np.frombuffer(Path(path).read_bytes(), dtype=np.uint8)
    if raw.size >= 768:
        raw = raw[-768:]
        rgb = np.stack([raw[:256], raw[256:512], raw[512:768]], axis=1)
    elif raw.size == 256:
        rgb = np.stack([raw] * 3, axis=1)
    else:
        raise ValueError(f"Unsupported Fiji LUT size for {path}: {raw.size} bytes")
    return mpl.colors.ListedColormap(rgb / 255.0, name=Path(path).stem)


def list_fiji_luts(prefix=""):
    return sorted(q.stem for q in Path(RAW_FIJI_LUT_DIR).glob("*.lut") if q.stem.startswith(prefix))


def raw_colormap():
    if RAW_USE_FIJI_LUT:
        p = Path(RAW_FIJI_LUT_NAME)
        if p.suffix.lower() != ".lut":
            p = Path(RAW_FIJI_LUT_DIR) / f"{RAW_FIJI_LUT_NAME}.lut"
        if not p.exists():
            raise FileNotFoundError(f"LUT {p} not found. Some available: {list_fiji_luts()[:40]}")
        if str(p) not in _RAW_LUT_CACHE:
            _RAW_LUT_CACHE[str(p)] = _read_fiji_lut(p)
        cmap = _RAW_LUT_CACHE[str(p)]
    else:
        cmap = mpl.colormaps[RAW_CMAP]
    return cmap.reversed() if RAW_REVERSE_LUT else cmap


def resolve_raw_tiff(seq):
    uid = str(seq["unique_id"].iloc[0])
    if uid in RAW_PATH_OVERRIDES:
        return Path(RAW_PATH_OVERRIDES[uid])
    m = seq.iloc[0]
    img = str(m["filename"]).replace("_tracked", "") + ".tif"
    cond = str(m["condition"]) if "condition" in seq.columns and pd.notna(m["condition"]) else None
    roots = [RAW_MASTER_DIR] if RAW_MASTER_DIR else []
    roots += [m[c] for c in ("foldername", "source_folder") if c in seq.columns and pd.notna(m[c]) and str(m[c])]
    cands = []
    for root in roots:
        root = Path(str(root))
        if cond:
            cands += [root / "data" / cond / img, root / cond / img]
        cands += [root / "data" / img, root / img]
    for c in cands:
        if c.exists():
            return c
    raise FileNotFoundError(f"raw TIFF not found for {uid}; tried: " + "; ".join(map(str, cands[:6])))


def _xy_span_um(x, y):
    return float(XY_SPAN_UM) if XY_SPAN_UM else max(np.ptp(x), np.ptp(y), 0.2) * (1 + 2 * XY_PAD_FRAC)


def _nice_bar(length, span):
    if length <= 0.45 * span:
        return float(length)
    nice = [b for b in (0.05, 0.1, 0.2, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0) if b <= 0.45 * span]
    return nice[-1] if nice else 0.45 * span


def _clamp_center(c, span, size):
    """Shift a crop centre so a span-wide box stays inside [-0.5, size - 0.5]."""
    if span >= size:
        return 0.5 * (size - 1)
    return float(np.clip(c, span / 2 - 0.5, size - 0.5 - span / 2))


def prepare_raw_panel(seq, inst, data):
    import tifffile

    path = resolve_raw_tiff(seq)
    px = float(PIXEL_SIZE_UM)
    frames = inst["frame"].to_numpy(dtype=float)
    f0, f1 = int(round(frames.min())), int(round(frames.max()))
    hx, hy = data["dense_x"] / px, data["dense_y"] / px
    xm, ym = 0.5 * (hx.min() + hx.max()), 0.5 * (hy.min() + hy.max())
    follow = str(RAW_FOV_MODE).lower() == "follow"
    if follow:
        span_px = float(RAW_FOLLOW_BOX_UM) / px
        pad = 0.5 * span_px + 2
        x0, x1, y0, y1 = hx.min() - pad, hx.max() + pad, hy.min() - pad, hy.max() + pad
    else:
        if isinstance(RAW_SPAN_UM, str) and RAW_SPAN_UM == "match_xy":
            span_px = _xy_span_um(data["dense_x"], data["dense_y"]) / px
        elif RAW_SPAN_UM is None:
            span_px = max(np.ptp(hx), np.ptp(hy)) + 2 * float(RAW_PAD_UM) / px
        else:
            span_px = float(RAW_SPAN_UM) / px

    with tifffile.TiffFile(str(path)) as tif:
        n_pages = len(tif.pages)
        H, W = tif.pages[0].shape[-2:]
        if not follow:
            xm, ym = _clamp_center(xm, span_px, W), _clamp_center(ym, span_px, H)
            x0, x1 = xm - span_px / 2 - 1, xm + span_px / 2 + 1
            y0, y1 = ym - span_px / 2 - 1, ym + span_px / 2 + 1
        cx0, cx1 = max(0, int(np.floor(x0))), min(W, int(np.ceil(x1)) + 1)
        cy0, cy1 = max(0, int(np.floor(y0))), min(H, int(np.ceil(y1)) + 1)
        if cx1 <= cx0 or cy1 <= cy0:
            raise ValueError(f"track lies outside the {W}x{H} raw image")
        idx = np.arange(f0, f1 + 1)
        if n_pages > f1:
            stack = np.empty((len(idx), cy1 - cy0, cx1 - cx0), dtype=np.float32)
            for i in range(0, len(idx), 256):
                chunk = np.asarray(tif.asarray(key=idx[i:i + 256].tolist()))
                if chunk.ndim == 2:
                    chunk = chunk[None]
                stack[i:i + len(chunk)] = chunk[..., cy0:cy1, cx0:cx1]
        else:
            full = np.asarray(tif.asarray())
            if full.ndim < 3 or full.shape[0] <= f1:
                raise ValueError(f"{path.name} has shape {full.shape}; no frame {f1}")
            stack = full[f0:f1 + 1, cy0:cy1, cx0:cx1].astype(np.float32)

    if int(RAW_TEMPORAL_AVG) > 1:
        stack = uniform_filter1d(stack, size=int(RAW_TEMPORAL_AVG), axis=0, mode="nearest")
    if RAW_CONTRAST_LIMITS is not None:
        lo, hi = (float(v) for v in RAW_CONTRAST_LIMITS)
    elif RAW_CONTRAST_PER_FRAME:
        lo = np.percentile(stack, RAW_CONTRAST_PERCENTILES[0], axis=(1, 2), keepdims=True)
        hi = np.percentile(stack, RAW_CONTRAST_PERCENTILES[1], axis=(1, 2), keepdims=True)
    else:
        lo, hi = np.percentile(stack, RAW_CONTRAST_PERCENTILES)
    stack = np.clip((stack - lo) / np.maximum(hi - lo, 1e-9), 0.0, 1.0)
    if float(RAW_GAMMA) != 1.0:
        stack = stack ** float(RAW_GAMMA)
    if RAW_INVERT:
        stack = 1.0 - stack
    return {
        "path": path, "stack": stack.astype(np.float32), "f0": f0,
        "extent": (cx0 - 0.5, cx1 - 0.5, cy1 - 0.5, cy0 - 0.5),
        "span_px": span_px, "center": (xm, ym), "follow": follow, "img_wh": (W, H),
    }


class LiveStateSoftmaxMovie:
    def __init__(self, data, taus, flash_w, flash_lab, seq, title=None, inst=None):
        self.d, self.taus, self.flash_w, self.flash_lab = data, taus, flash_w, flash_lab
        self.n_frames = len(taus)
        observed = mixture_state_order(data["states"])
        self.colors = {s: mixture_state_color(s) for s in set(observed) | set(PURE_STATES)}
        self.gold = np.array(FLASH_GOLD, dtype=float)

        self.raw, self.raw_path, self.raw_error = None, None, None
        if RAW_PANEL:
            try:
                if inst is None:
                    raise ValueError("RAW_PANEL needs inst= (the instant rows of this track)")
                self.raw = prepare_raw_panel(seq, inst, data)
                self.raw_path = str(self.raw["path"])
            except Exception as exc:
                if str(RAW_ON_MISSING).lower() == "error":
                    raise
                self.raw_error = f"{type(exc).__name__}: {exc}"
                print("[raw panel skipped]", self.raw_error)
        self.xy_invert = bool(XY_INVERT_Y or (self.raw is not None and RAW_FORCE_XY_IMAGE_ORIENTATION))

        base_w, fig_h = MOVIE_FIGSIZE
        rib_h = 0.12 if SHOW_ASSIGNMENT_RIBBON else 0.0
        if self.raw is None:
            fig = plt.figure(figsize=MOVIE_FIGSIZE, dpi=MOVIE_DPI)
            gs = fig.add_gridspec(
                2, 2, width_ratios=[1.0, 1.55], height_ratios=[1.0, max(rib_h, 1e-3)],
                left=0.03, right=0.80, bottom=0.12, top=0.88, wspace=0.14, hspace=0.08,
            )
            ax_raw = None
            ax_xy = fig.add_subplot(gs[:, 0])
            ax_soft = fig.add_subplot(gs[0, 1])
            ax_rib = fig.add_subplot(gs[1, 1], sharex=ax_soft) if SHOW_ASSIGNMENT_RIBBON else None
        else:
            fig_w = int(np.ceil((base_w + float(RAW_PANEL_WIDTH_IN)) * MOVIE_DPI / 2.0)) * 2 / MOVIE_DPI
            fig_h = int(np.ceil(fig_h * MOVIE_DPI / 2.0)) * 2 / MOVIE_DPI
            fig = plt.figure(figsize=(fig_w, fig_h), dpi=MOVIE_DPI)
            gs = fig.add_gridspec(
                2, 3, width_ratios=[1.0, 1.0, 1.55], height_ratios=[1.0, max(rib_h, 1e-3)],
                left=0.02, right=1.0 - 0.20 * base_w / fig_w, bottom=0.12, top=0.88, wspace=0.12, hspace=0.08,
            )
            ax_raw = fig.add_subplot(gs[:, 0])
            ax_xy = fig.add_subplot(gs[:, 1])
            ax_soft = fig.add_subplot(gs[0, 2])
            ax_rib = fig.add_subplot(gs[1, 2], sharex=ax_soft) if SHOW_ASSIGNMENT_RIBBON else None
        fig.patch.set_facecolor("white")
        self.fig, self.ax_raw, self.ax_xy, self.ax_soft, self.ax_rib = fig, ax_raw, ax_xy, ax_soft, ax_rib

        self._setup_xy()
        if self.raw is not None:
            self._setup_raw()
        self._setup_soft()
        if ax_rib is not None:
            self._setup_ribbon()
        self._setup_legends(observed)
        if title:
            fig.suptitle(title, fontsize=10, y=0.975)
        self.draw(0)

    def _setup_xy(self):
        ax, d = self.ax_xy, self.d
        x, y = d["dense_x"], d["dense_y"]
        pts = np.column_stack([x, y])
        self.pts = pts
        self.segs = np.stack([pts[:-1], pts[1:]], axis=1) if len(pts) > 1 else np.zeros((0, 2, 2))
        self.seg_rgba = np.array([mpl.colors.to_rgba(self.colors[s]) for s in d["seg_state"]]).reshape(-1, 4)
        self.glow = LineCollection([], capstyle="round", joinstyle="round", zorder=3)
        self.trail = LineCollection([], linewidths=TRACK_LINEWIDTH, capstyle="round", joinstyle="round", zorder=4)
        ax.add_collection(self.glow)
        ax.add_collection(self.trail)
        self.ring = ax.scatter([x[0]], [y[0]], s=[0], facecolors="none", edgecolors=[(*self.gold, 0)], linewidths=2.0, zorder=5)
        self.head = ax.scatter([x[0]], [y[0]], s=[HEAD_SIZE_PT ** 2], edgecolors="k", linewidths=0.6, zorder=6)

        inv = self.xy_invert
        xm, ym = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
        span = _xy_span_um(x, y)
        ax.set_xlim(xm - span / 2, xm + span / 2)
        ax.set_ylim((ym + span / 2, ym - span / 2) if inv else (ym - span / 2, ym + span / 2))
        ax.set_aspect("equal", adjustable="box")
        ax.set_xticks([])
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_color("0.75")
        bar = _nice_bar(float(SCALE_BAR_UM), span)
        bx = xm - span / 2 + 0.07 * span
        by = ym - span / 2 + 0.07 * span if not inv else ym + span / 2 - 0.07 * span
        ax.plot([bx, bx + bar], [by, by], color="k", lw=2.2, solid_capstyle="butt", zorder=7)
        ax.text(bx + bar / 2, by + (0.025 * span) * (-1 if inv else 1), f"{bar:g} \u00b5m",
                ha="center", va="bottom" if not inv else "top", fontsize=8, zorder=7)
        self.txt_time = ax.text(0.03, 0.97, "", transform=ax.transAxes, ha="left", va="top", fontsize=10, family="monospace")
        self.txt_state = ax.text(0.03, 0.895, "", transform=ax.transAxes, ha="left", va="top", fontsize=10, fontweight="bold")
        ax.set_title("track, coloured by assignment", fontsize=9)

    def _setup_raw(self):
        ax, r = self.ax_raw, self.raw
        px = float(PIXEL_SIZE_UM)
        ax.set_facecolor("white" if RAW_INVERT else "black")
        self.raw_im = ax.imshow(r["stack"][0], cmap=raw_colormap(), vmin=0.0, vmax=1.0, extent=r["extent"],
                                origin="upper", interpolation=RAW_INTERPOLATION, zorder=0)
        self.raw_glow = LineCollection([], capstyle="round", joinstyle="round", zorder=3)
        self.raw_trail = LineCollection([], linewidths=RAW_TRACK_LINEWIDTH, capstyle="round", joinstyle="round", zorder=4)
        ax.add_collection(self.raw_glow)
        ax.add_collection(self.raw_trail)
        p0 = self.pts[0] / px
        style = str(RAW_HEAD_STYLE).lower()
        self.raw_head = ax.scatter([p0[0]], [p0[1]], s=[RAW_HEAD_SIZE_PT ** 2],
                                   facecolors="none" if style == "ring" else "w",
                                   edgecolors="w", linewidths=1.6 if style == "ring" else 0.6, zorder=6)
        self.raw_head.set_visible(style != "none")
        self.raw_veil = Rectangle((0, 0), 1, 1, transform=ax.transAxes, facecolor="white",
                                  alpha=0.0, linewidth=0, zorder=2)
        ax.add_patch(self.raw_veil)
        span = r["span_px"]
        xm, ym = r["center"]
        ax.set_xlim(xm - span / 2, xm + span / 2)
        ax.set_ylim(ym + span / 2, ym - span / 2)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        if str(RAW_TEXT_COLOR) != "auto":
            txt_col = RAW_TEXT_COLOR
        else:
            rgb = np.asarray(self.raw_im.cmap(float(np.median(r["stack"]))))[:3]
            txt_col = "black" if float(rgb @ [0.299, 0.587, 0.114]) > 0.5 else "white"
        bar_um = _nice_bar(float(RAW_SCALE_BAR_UM), span * px)
        ax.add_artist(AnchoredSizeBar(
            ax.transData, bar_um / px, f"{bar_um:g} \u00b5m", "lower left", pad=0.5, borderpad=0.4,
            color=txt_col, frameon=False, size_vertical=max(span * 0.012, 0.3),
            fontproperties=fm.FontProperties(size=8),
        ))
        self.raw_txt_time = ax.text(0.03, 0.97, "", transform=ax.transAxes, ha="left", va="top",
                                    fontsize=10, family="monospace", color=txt_col) if RAW_SHOW_TIMER else None
        ax.set_title("raw data + track", fontsize=9)

    def _draw_raw(self, tau, w, segs, cols, head, scol):
        r, px = self.raw, float(PIXEL_SIZE_UM)
        st = r["stack"]
        pos = float(np.clip(tau / FRAME_INTERVAL, 0, len(st) - 1))
        if RAW_FRAME_BLEND:
            i0 = int(np.floor(pos))
            i1 = min(i0 + 1, len(st) - 1)
            a = pos - i0
            img = st[i0] * (1.0 - a) + st[i1] * a
        else:
            img = st[int(round(pos))]
        self.raw_im.set_data(img)

        if RAW_SHOW_TRACK and len(segs):
            rs = np.asarray(segs) / px
            rc = np.array(cols, dtype=float, copy=True)
            if str(RAW_TRAIL_MODE).lower() == "tail":
                seg_t = self.d["dense_t"][:len(rs)]
                keep = seg_t >= tau - float(RAW_TAIL_S)
                rs, rc, seg_t = rs[keep], rc[keep], seg_t[keep]
                fade = np.clip(1 - (tau - seg_t) / max(float(RAW_TAIL_S), 1e-9), 0.05, 1) if RAW_TAIL_FADE else 1.0
                rc[:, 3] = RAW_TRACK_ALPHA * fade
            else:
                rc[:, 3] = RAW_TRACK_ALPHA
            self.raw_trail.set_segments(rs)
            self.raw_trail.set_color(rc)
            if RAW_FLASH_GLOW and w > 0 and len(rs):
                self.raw_glow.set_segments(rs)
                self.raw_glow.set_color([(*self.gold, 0.18 + 0.45 * w)])
                self.raw_glow.set_linewidth(RAW_TRACK_LINEWIDTH * (2.4 + 2.6 * w))
                self.raw_glow.set_visible(True)
            else:
                self.raw_glow.set_visible(False)
        else:
            self.raw_trail.set_segments([])
            self.raw_glow.set_visible(False)

        hpx = np.asarray(head) / px
        self.raw_head.set_offsets([hpx])
        self.raw_head.set_sizes([(RAW_HEAD_SIZE_PT * (1.0 + 0.6 * w)) ** 2])
        if str(RAW_HEAD_STYLE).lower() == "ring":
            self.raw_head.set_edgecolor([scol])
        else:
            self.raw_head.set_facecolor([scol])
        self.raw_veil.set_alpha(float(RAW_FLASH_WHITEN) * w)
        if r["follow"]:
            span = r["span_px"]
            fx = _clamp_center(hpx[0], span, r["img_wh"][0])
            fy = _clamp_center(hpx[1], span, r["img_wh"][1])
            self.ax_raw.set_xlim(fx - span / 2, fx + span / 2)
            self.ax_raw.set_ylim(fy + span / 2, fy - span / 2)
        if self.raw_txt_time is not None:
            self.raw_txt_time.set_text(f"t = {tau:5.2f} s")

    def _setup_soft(self):
        ax, d = self.ax_soft, self.d
        t, P = d["t_soft"], d["P_soft"]
        cum = np.cumsum(P, axis=0)
        self.cum = cum
        self.clipped = []
        lower = np.zeros_like(t)
        for i, s in enumerate(PURE_STATES):
            col = BEHAVIOR_STATE_PALETTE[s]
            if SHOW_FUTURE_GHOST:
                ax.fill_between(t, lower, cum[i], color=col, alpha=0.10, linewidth=0, zorder=1)
            poly = ax.fill_between(t, lower, cum[i], color=col, alpha=0.92, linewidth=0, zorder=2)
            line, = ax.plot(t, cum[i], color=col, lw=1.5, solid_capstyle="round", zorder=3)
            self.clipped += [poly, line]
            lower = cum[i]
        if MARK_CHANGES_ON_AREA:
            for t_c, _ in d["changes"]:
                self.clipped.append(ax.axvline(t_c, color="k", lw=0.8, ls=(0, (2, 2)), alpha=0.55, zorder=4))
        self.cursor = ax.axvline(0.0, color="k", lw=1.2, zorder=6)
        self.tips = ax.scatter(np.zeros(len(PURE_STATES)), cum[:, 0],
                               c=[BEHAVIOR_STATE_PALETTE[s] for s in PURE_STATES],
                               s=26, edgecolors="k", linewidths=0.6, zorder=7)
        ax.set_xlim(0.0, d["t_end"])
        ax.set_ylim(0.0, 1.0)
        ax.set_ylabel("P(state)")
        ax.set_title("transformer softmax vs time", fontsize=9)
        ax.grid(False)
        if self.ax_rib is not None:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel("time (s)")

    def _setup_ribbon(self):
        ax, d = self.ax_rib, self.d
        t, st = d["t_frame"], d["states"]
        edges = np.append(t, d["t_end"] + (t[-1] - t[-2] if len(t) > 1 else FRAME_INTERVAL))
        start = 0
        for i in range(1, len(st) + 1):
            if i == len(st) or st[i] != st[start]:
                r = ax.broken_barh([(edges[start], edges[i] - edges[start])], (0, 1),
                                   facecolors=self.colors[st[start]], linewidth=0)
                self.clipped.append(r)
                start = i
        self.rib_cursor = ax.axvline(0.0, color="k", lw=1.2, zorder=6)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_xlabel("time (s)")
        ax.grid(False)

    def _setup_legends(self, observed):
        pure = [Patch(facecolor=BEHAVIOR_STATE_PALETTE[s], label=s) for s in PURE_STATES]
        self.ax_soft.legend(handles=pure, loc="upper left", bbox_to_anchor=(1.02, 1.0),
                            frameon=False, fontsize=8, title="softmax", title_fontsize=8)
        extra = [s for s in observed if s not in PURE_STATES]
        if extra and self.ax_rib is not None:
            handles = [Patch(facecolor=self.colors[s], label=s) for s in extra]
            self.ax_rib.legend(handles=handles, loc="lower left", bbox_to_anchor=(1.02, 0.0),
                               frameon=False, fontsize=7.5, title="mixed assignments", title_fontsize=8)

    def _state_at(self, tau):
        i = int(np.clip(np.searchsorted(self.d["t_frame"], tau, side="right") - 1, 0, len(self.d["states"]) - 1))
        return str(self.d["states"][i])

    def _head_at(self, tau):
        dt = self.d["dense_t"]
        m = int(np.searchsorted(dt, tau, side="right"))
        if m <= 0:
            return 0, self.pts[0]
        if m >= len(dt):
            return len(dt) - 1, self.pts[-1]
        span = dt[m] - dt[m - 1]
        s = 0.0 if span <= 0 else (tau - dt[m - 1]) / span
        return m - 1, self.pts[m - 1] + (self.pts[m] - self.pts[m - 1]) * s

    def draw(self, k):
        tau = float(self.taus[k])
        w = float(self.flash_w[k])
        j, head = self._head_at(tau)

        segs = self.segs[:j]
        cols = self.seg_rgba[:j]
        if j < len(self.segs):
            segs = np.concatenate([segs, [[self.pts[j], head]]], axis=0)
            cols = np.concatenate([cols, self.seg_rgba[j:j + 1]], axis=0)
        self.trail.set_segments(segs)
        self.trail.set_color(cols)
        if w > 0:
            self.glow.set_segments(segs)
            self.glow.set_color([(*self.gold, 0.18 + 0.45 * w)])
            self.glow.set_linewidth(TRACK_LINEWIDTH * (2.4 + 2.6 * w))
            self.glow.set_visible(True)
        else:
            self.glow.set_visible(False)

        state = self._state_at(tau)
        scol = self.colors.get(state, "#999999")
        size = HEAD_SIZE_PT * (1.0 + HEAD_PULSE_SCALE * w)
        self.head.set_offsets([head])
        self.head.set_sizes([size ** 2])
        self.head.set_facecolor([scol])
        self.ring.set_offsets([head])
        self.ring.set_sizes([(size * 1.6) ** 2 if w > 0 else 0.0])
        self.ring.set_edgecolor([(*self.gold, 0.35 + 0.55 * w)])
        self.ring.set_facecolor([(*self.gold, 0.08 + 0.15 * w)])
        if self.raw is not None:
            self._draw_raw(tau, w, segs, cols, head, scol)

        self.txt_time.set_text(f"t = {tau:5.2f} s")
        lab = self.flash_lab[k] if (w > 0 and self.flash_lab[k] is not None) else state
        self.txt_state.set_text(lab)
        self.txt_state.set_color(scol)

        clip_w = max(tau, 1e-9)
        for art in self.clipped:
            ax = art.axes
            art.set_clip_path(Rectangle((0.0, -1.0), clip_w, 3.0, transform=ax.transData))
        self.cursor.set_xdata([tau, tau])
        if self.ax_rib is not None:
            self.rib_cursor.set_xdata([tau, tau])
        tips_y = np.array([np.interp(tau, self.d["t_soft"], self.cum[i]) for i in range(len(PURE_STATES))])
        self.tips.set_offsets(np.column_stack([np.full(len(tips_y), tau), tips_y]))
        return []

    def save_mp4(self, path, crf=17):
        writer = animation.FFMpegWriter(fps=MOVIE_FPS, codec="libx264",
                                        extra_args=["-pix_fmt", "yuv420p", "-preset", "slow", "-crf", str(int(crf))])
        self._save(path, writer)

    def save_gif(self, path):
        self._save(path, animation.PillowWriter(fps=MOVIE_FPS))

    def _save(self, path, writer):
        anim = animation.FuncAnimation(self.fig, self.draw, frames=self.n_frames, blit=False, repeat=False)
        t0 = time.time()
        anim.save(str(path), writer=writer, dpi=MOVIE_DPI, savefig_kwargs={"facecolor": "white"})
        print(f"Saved {path}  ({self.n_frames} frames, {time.time() - t0:.1f} s)")

    def save_still(self, k, path):
        self.draw(k)
        self.fig.savefig(path, dpi=MOVIE_DPI if str(path).endswith(".png") else 300, facecolor="white")
        print("Saved:", path)


meta = seq.iloc[0]
TITLE = "   ".join(
    str(v) for v in [
        PICK_UID,
        *[f"{c}={meta[c]}" for c in ("mol", "geno", "type", "location") if c in seq.columns],
        f"{len(seq)} windows",
    ]
)
live = LiveStateSoftmaxMovie(movie_data, taus, flash_w, flash_lab, seq, title=TITLE, inst=inst)

TAG = f"{PICK_UID}_{PICK_CONTENT}_seed{PICK_RANDOM_SEED}".replace("/", "_")
OUT_DIR = MOVIE_DIR / TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
preview_ks = sorted({int(q * (live.n_frames - 1)) for q in (0.25, 0.5, 0.75, 1.0)})
if FLASH_ON_CHANGE and np.any(flash_w > 0):
    preview_ks.insert(1, int(np.argmax(flash_w)))
for k in preview_ks:
    p = OUT_DIR / f"still_k{k:05d}.png"
    live.save_still(k, p)
    display(Image(filename=str(p), width=900))
live.save_still(live.n_frames - 1, OUT_DIR / "final_frame.svg")
plt.close(live.fig)

## Render the movie

Writes `live_state_softmax.mp4` (H.264, smooth) and optionally a GIF, plus a per-frame CSV of
the track + assignment + softmax at every localisation.

In [ ]:
live = LiveStateSoftmaxMovie(movie_data, taus, flash_w, flash_lab, seq, title=TITLE, inst=inst)
mp4_path = OUT_DIR / "live_state_softmax.mp4"
gif_path = OUT_DIR / "live_state_softmax.gif"
if MOVIE_WRITE_MP4:
    live.save_mp4(mp4_path)
if MOVIE_WRITE_GIF:
    live.save_gif(gif_path)
plt.close(live.fig)

frame_table = pd.DataFrame({
    "unique_id": PICK_UID,
    "frame": inst["frame"].to_numpy(),
    "t_s": movie_data["t_frame"],
    "x_um": inst["x_um"].to_numpy(),
    "y_um": inst["y_um"].to_numpy(),
    STATE_COL: movie_data["states"],
})
for i, s in enumerate(PURE_STATES):
    frame_table[f"prob_{s}_smooth"] = np.interp(movie_data["t_frame"], movie_data["t_soft"], movie_data["P_soft"][i])
frame_table.to_csv(OUT_DIR / "per_frame_track_softmax.csv", index=False)
seq.to_csv(OUT_DIR / "windows.csv", index=False)
print("Output folder:", OUT_DIR)

if MOVIE_WRITE_MP4 and mp4_path.exists():
    display(Video(str(mp4_path), embed=True, width=960))
elif MOVIE_WRITE_GIF and gif_path.exists():
    display(Image(filename=str(gif_path)))

## Batch — many movies per molecule to choose from

Run every cell above first (the single-movie render cell can be skipped). Then:

1. Leave `BATCH_SMOKE_TEST = True` and run the two batch cells. That makes **1 movie per
   (molecule, content)** = 15 low-res movies in `batch_smoke/`, in a few minutes.
2. Check the contact sheet + a couple of MP4s.
3. Set `BATCH_SMOKE_TEST = False` and run again for the full plan (22 per molecule).

The run is resumable: with `BATCH_SKIP_EXISTING = True`, an interrupted run picks up where it
stopped (same seed -> same track selection). `batch_manifest.csv` is rewritten after every
movie, and one failing track is logged and skipped instead of stopping the run.

**Selection.** Tracks are picked by molecule only (`mol` + ungated drop); set
`BATCH_USE_DATASET_FILTER = True` to also apply the top filter cell. Each content type walks a
list of tiers, strict first, and only relaxes when the strict tier runs out of tracks
(`tier` in the manifest says which one was used):

- `mixed`: >= 3 distinct assignments and >= 3 switches, 8+ continuous windows, then relaxed.
- state focus (`bound` / `confined` / `transport` / `exploratory`): the state must cover at
  least `min_frac` of the windows, **and** the track must switch at least `min_switches` times.
  Only the last tier allows non-switching tracks, so the state is always present and switching
  is preferred.
- Long continuous confined / transport tracks are rare after the ungated drop, so later tiers
  accept shorter tracks (`min_windows`) and one or two missing windows (`max_gap`).

The smoke test also prints how many tracks the **full** plan would find, so shortfalls show up
before the long run.

Within a tier, one track is drawn at random from the top `n * BATCH_TOP_MULT` candidates.
Each track appears only once per molecule, and at most `BATCH_MAX_PER_FILE` tracks come from
one acquisition file.

In [ ]:
# =============================================================================
# BATCH KNOBS
# =============================================================================
BATCH_SMOKE_TEST = False           # True: 1 movie per (molecule, content), low-res, batch_smoke/
BATCH_LOW_RES = True               # full plan, but rendered at BATCH_SMOKE_DPI / BATCH_SMOKE_FPS (fast preview)
BATCH_RUN_TAG = "htt_exp_to_bc"    # output folder name: batch_<tag> (batch_lowres_<tag> when BATCH_LOW_RES)
BATCH_MOLECULES = ["HTT"]
# Content types: "mixed" | "bound" | "confined" | "transport" | "exploratory"
#   | "exp_bc" = exploratory -> bound/confined switch, preferring tracks with all three states
BATCH_PLAN = {"exp_bc": 200}
BATCH_USE_DATASET_FILTER = False    # True = also apply DATASET_FILTER / FILENAME_CONTAINS / EXTRA_FILTER
BATCH_SEED = 21
BATCH_TOP_MULT = 3
BATCH_MAX_PER_FILE = 3
BATCH_SKIP_EXISTING = True
BATCH_SMOKE_DPI = 75                # figsize * dpi must stay even for H.264 (11.2 x 4.8 in -> 840 x 360)
BATCH_SMOKE_FPS = 15

# Tiers are tried strict -> loose until the plan is filled.
# max_gap = largest time_window step allowed inside a track (1 = continuous). One missing
# window (gap 2) leaves no frame gap, because neighbouring windows overlap by 30 frames.
BATCH_MIXED_TIERS = [
    dict(min_distinct=3, min_switches=3, exclude_unresolved=True, min_windows=8, max_gap=1),
    dict(min_distinct=3, min_switches=2, exclude_unresolved=False, min_windows=6, max_gap=1),
    dict(min_distinct=3, min_switches=2, exclude_unresolved=False, min_windows=8, max_gap=2),
    dict(min_distinct=2, min_switches=2, exclude_unresolved=False, min_windows=5, max_gap=2),
    dict(min_distinct=2, min_switches=1, exclude_unresolved=False, min_windows=4, max_gap=3),
]
BATCH_STATE_TIERS = [
    dict(min_frac=0.35, min_switches=2, exclude_unresolved=True, min_windows=8, max_gap=1),
    dict(min_frac=0.25, min_switches=1, exclude_unresolved=True, min_windows=6, max_gap=1),
    dict(min_frac=0.25, min_switches=1, exclude_unresolved=True, min_windows=8, max_gap=2),
    dict(min_frac=0.20, min_switches=1, exclude_unresolved=False, min_windows=5, max_gap=2),
    dict(min_frac=0.20, min_switches=1, exclude_unresolved=False, min_windows=4, max_gap=3),
    dict(min_frac=0.20, min_switches=0, exclude_unresolved=False, min_windows=4, max_gap=3),
]
# exp_bc: every tier needs >= min_exp_to_bc switches from an exploratory-containing window
# (exploratory, confined_exploratory, ...) into a bound/confined window without exploratory.
# require_all3 = bound, confined and exploratory each appear somewhere in the track.
BATCH_EXP_BC_TIERS = [
    dict(require_all3=True, min_exp_to_bc=1, exclude_unresolved=True, min_windows=8, max_gap=1),
    dict(require_all3=True, min_exp_to_bc=1, exclude_unresolved=True, min_windows=6, max_gap=2),
    dict(require_all3=True, min_exp_to_bc=1, exclude_unresolved=False, min_windows=5, max_gap=2),
    dict(require_all3=True, min_exp_to_bc=1, exclude_unresolved=False, min_windows=4, max_gap=3),
    dict(require_all3=False, min_exp_to_bc=1, exclude_unresolved=True, min_windows=6, max_gap=2),
    dict(require_all3=False, min_exp_to_bc=1, exclude_unresolved=False, min_windows=4, max_gap=3),
]

In [ ]:
from collections import Counter


def add_transition_columns(pool, windowed):
    base = pl.col(STATE_COL).cast(pl.Utf8).str.replace("^unresolved_", "")
    d = (
        windowed.select(["unique_id", "time_window", STATE_COL])
        .with_columns(pl.col("unique_id").cast(pl.Utf8))
        .sort(["unique_id", "time_window"])
        .with_columns(
            base.str.contains("exploratory").alias("_e"),
            base.str.contains("bound").alias("_b"),
            base.str.contains("confined").alias("_c"),
        )
        .with_columns(
            (pl.col("_e").shift(1).over("unique_id").fill_null(False) & ~pl.col("_e") & (pl.col("_b") | pl.col("_c")))
            .alias("_e2bc")
        )
    )
    t = d.group_by("unique_id").agg(
        pl.col("_e2bc").sum().alias("n_exp_to_bc"),
        (pl.col("_e").any() & pl.col("_b").any() & pl.col("_c").any()).alias("has_all3"),
    ).to_pandas()
    return pool.merge(t, on="unique_id", how="left").fillna({"n_exp_to_bc": 0, "has_all3": False})


def select_batch_tracks(pool, content, n, used, per_file, rng):
    base = pool
    if PICK_MAX_WINDOWS is not None:
        base = base.loc[base["n_windows"] <= int(PICK_MAX_WINDOWS)]
    if content == "exp_bc":
        tiers = BATCH_EXP_BC_TIERS
    else:
        tiers = BATCH_MIXED_TIERS if content == "mixed" else BATCH_STATE_TIERS
    picked = []
    for tier_i, tier in enumerate(tiers):
        need = n - len(picked)
        if need <= 0:
            break
        df = base.loc[~base["unique_id"].isin(used)]
        df = df.loc[(df["n_windows"] >= int(tier["min_windows"])) & (df["max_tw_gap"] <= int(tier["max_gap"]))]
        if tier.get("exclude_unresolved"):
            df = df.loc[~df["has_unresolved"]]
        if content == "exp_bc":
            df = df.loc[df["n_exp_to_bc"] >= int(tier["min_exp_to_bc"])]
            if tier.get("require_all3"):
                df = df.loc[df["has_all3"].astype(bool)]
            score = (
                (1.0 + np.log1p(df["n_exp_to_bc"]))
                * (1.0 + df["assign_entropy"])
                * (1.0 + df["has_all3"].astype(float))
            )
            df = df.assign(state_score=score).sort_values("state_score", ascending=False)
        elif content == "mixed":
            df = df.loc[(df["n_distinct"] >= tier["min_distinct"]) & (df["n_switches"] >= tier["min_switches"])]
            df = df.sort_values(["mix_score", "n_distinct", "n_switches"], ascending=False)
        else:
            frac = f"frac_{content}"
            df = df.loc[(df[frac] >= tier["min_frac"]) & (df["n_switches"] >= tier["min_switches"])]
            score = df[frac] * (1.0 + df["assign_entropy"]) * (1.0 + np.log1p(df["n_switches"]))
            df = df.assign(state_score=score).sort_values("state_score", ascending=False)
        if df.empty:
            continue
        n_top = min(len(df), max(need * int(BATCH_TOP_MULT), need))
        order = list(rng.permutation(n_top)) + list(range(n_top, len(df)))
        for i in order:
            row = df.iloc[int(i)]
            if per_file[row["filename"]] >= int(BATCH_MAX_PER_FILE):
                continue
            picked.append({**row.to_dict(), "content": content, "tier": tier_i})
            used.add(row["unique_id"])
            per_file[row["filename"]] += 1
            if len(picked) >= n:
                break
    return picked


def load_instants(uids):
    lf = pl.scan_parquet(INSTANT_PARQUET)
    names = lf.collect_schema().names()
    cols = [c for c in ("unique_id", "window_uid", "frame", "time_s", "x_um", "y_um", "x", "y") if c in names]
    df = lf.filter(pl.col("unique_id").cast(pl.Utf8).is_in([str(u) for u in uids])).select(cols).collect().to_pandas()
    if "x_um" not in df.columns:
        df["x_um"] = df["x"].astype(float) * PIXEL_SIZE_UM
        df["y_um"] = df["y"].astype(float) * PIXEL_SIZE_UM
    return {
        str(uid): g.sort_values("frame").drop_duplicates("frame", keep="last").reset_index(drop=True)
        for uid, g in df.groupby("unique_id")
    }


def render_batch_movie(row, seq_b, inst_b, out_dir):
    data = build_movie_data(seq_b, inst_b)
    ta, fw, fl = build_timeline(data)
    meta_b = seq_b.iloc[0]
    title = "   ".join([
        f"{row['content']} #{row['rank']:02d}",
        str(row["unique_id"]),
        *[f"{c}={meta_b[c]}" for c in ("mol", "geno", "type", "location") if c in seq_b.columns],
        f"{len(seq_b)} windows",
    ])
    live_b = LiveStateSoftmaxMovie(data, ta, fw, fl, seq_b, title=title, inst=inst_b)
    try:
        live_b.save_still(len(ta) - 1, out_dir / "final_frame.png")
        live_b.save_mp4(out_dir / "live_state_softmax.mp4")
    finally:
        plt.close(live_b.fig)
    return len(ta), len(data["changes"]), data["t_end"]


plan = {k: (1 if BATCH_SMOKE_TEST else int(v)) for k, v in BATCH_PLAN.items()}
_low_res = BATCH_SMOKE_TEST or BATCH_LOW_RES
_tag = f"_{BATCH_RUN_TAG}" if BATCH_RUN_TAG else ""
BATCH_ROOT = MOVIE_DIR / (f"batch_smoke{_tag}" if BATCH_SMOKE_TEST else f"batch{'_lowres' if _low_res else ''}{_tag}")
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = BATCH_ROOT / "batch_manifest.csv"
RANK_W = max(2, len(str(max(plan.values()))))
_mode = "SMOKE TEST" if BATCH_SMOKE_TEST else ("FULL RUN (low-res)" if _low_res else "FULL RUN")
print(f"{_mode}: {plan} x {BATCH_MOLECULES} -> {BATCH_ROOT}")

_saved_movie_settings = (MOVIE_DPI, MOVIE_FPS)
if _low_res:
    MOVIE_DPI, MOVIE_FPS = BATCH_SMOKE_DPI, BATCH_SMOKE_FPS

manifest_rows = []
batch_t0 = time.time()
try:
    selections = {}
    for mol in BATCH_MOLECULES:
        flt = {**DATASET_FILTER, "mol": [mol]} if BATCH_USE_DATASET_FILTER else {"mol": [mol]}
        windowed_mol, _ = apply_dataset_filter(windowed_full, flt, use_extra=BATCH_USE_DATASET_FILTER)
        pool_mol = add_transition_columns(build_track_pool(windowed_mol), windowed_mol)
        if BATCH_SMOKE_TEST:
            rng_dry, used_dry, pf_dry = np.random.default_rng(int(BATCH_SEED)), set(), Counter()
            dry = {c: select_batch_tracks(pool_mol, c, int(v), used_dry, pf_dry, rng_dry) for c, v in BATCH_PLAN.items()}
            print(f"  {mol}: FULL plan availability -> "
                  + ", ".join(f"{c} {len(dry[c])}/{v}" for c, v in BATCH_PLAN.items()))
        rng = np.random.default_rng(int(BATCH_SEED))
        used, per_file = set(), Counter()
        rows = []
        for content, n in plan.items():
            got = select_batch_tracks(pool_mol, content, n, used, per_file, rng)
            for rank, r in enumerate(got, start=1):
                r["rank"] = rank
            rows += got
            tiers = Counter(r["tier"] for r in got)
            flag = "" if len(got) == n else f"   <-- only {len(got)} available"
            print(f"  {mol:8s} {content:12s} {len(got)}/{n}  tiers={dict(sorted(tiers.items()))}{flag}")
        selections[mol] = (windowed_mol, rows)

    n_total = sum(len(rows) for _, rows in selections.values())
    done = 0
    for mol, (windowed_mol, rows) in selections.items():
        insts = load_instants([r["unique_id"] for r in rows])
        for r in rows:
            done += 1
            uid = str(r["unique_id"])
            out_dir = BATCH_ROOT / mol / f"{r['content']}_{r['rank']:0{RANK_W}d}_{uid}"
            mp4 = out_dir / "live_state_softmax.mp4"
            rec = {
                "mol": mol, "content": r["content"], "rank": r["rank"], "tier": r["tier"], "unique_id": uid,
                "filename": r["filename"], "n_windows": r["n_windows"], "max_tw_gap": r["max_tw_gap"],
                "n_distinct": r["n_distinct"],
                "n_switches": r["n_switches"], "has_unresolved": r["has_unresolved"],
                "n_exp_to_bc": int(r["n_exp_to_bc"]), "has_all3": bool(r["has_all3"]),
                **{f"frac_{s}": round(float(r[f"frac_{s}"]), 3) for s in PURE_STATES},
                "mp4": str(mp4), "status": "", "error": "", "render_s": np.nan,
            }
            t0 = time.time()
            try:
                if BATCH_SKIP_EXISTING and mp4.exists():
                    rec["status"] = "skipped_existing"
                else:
                    out_dir.mkdir(parents=True, exist_ok=True)
                    seq_b = (
                        windowed_mol.filter(pl.col("unique_id").cast(pl.Utf8) == uid)
                        .sort("time_window").to_pandas().reset_index(drop=True)
                    )
                    if uid not in insts:
                        raise ValueError("no instant rows")
                    n_fr, n_ch, t_end = render_batch_movie(r, seq_b, insts[uid], out_dir)
                    seq_b.to_csv(out_dir / "windows.csv", index=False)
                    rec.update(status="ok", movie_frames=n_fr, n_changes=n_ch, track_s=round(t_end, 2))
            except Exception as exc:
                rec.update(status="failed", error=f"{type(exc).__name__}: {exc}")
                plt.close("all")
            rec["render_s"] = round(time.time() - t0, 1)
            manifest_rows.append(rec)
            pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
            elapsed = time.time() - batch_t0
            eta = elapsed / done * (n_total - done)
            print(f"[{done}/{n_total}] {mol} {r['content']} #{r['rank']:0{RANK_W}d} {uid}: {rec['status']} "
                  f"({rec['render_s']} s)  elapsed {elapsed / 60:.1f} min  ETA {eta / 60:.1f} min"
                  + (f"  {rec['error']}" if rec["error"] else ""))
finally:
    MOVIE_DPI, MOVIE_FPS = _saved_movie_settings

batch_manifest = pd.DataFrame(manifest_rows)
print(f"\nDone in {(time.time() - batch_t0) / 60:.1f} min.  Manifest: {manifest_path}")
display(batch_manifest.groupby(["mol", "status"]).size().unstack(fill_value=0))

In [ ]:
# Contact sheet of final frames per molecule, for choosing between movies.
CONTACT_COLS = 3
CONTACT_PER_PAGE = 24              # large batches are split into pages
CONTACT_SHOW_INLINE = True

manifest = pd.read_csv(BATCH_ROOT / "batch_manifest.csv")
for mol, g in manifest.groupby("mol", sort=False):
    g = g.loc[g["status"].isin(["ok", "skipped_existing"])]
    paths = [(Path(p).parent / "final_frame.png", r) for p, (_, r) in zip(g["mp4"], g.iterrows())]
    paths = [(p, r) for p, r in paths if p.exists()]
    if not paths:
        continue
    n_pages = int(np.ceil(len(paths) / CONTACT_PER_PAGE))
    for page in range(n_pages):
        chunk = paths[page * CONTACT_PER_PAGE:(page + 1) * CONTACT_PER_PAGE]
        n_rows = int(np.ceil(len(chunk) / CONTACT_COLS))
        fig, axes = plt.subplots(n_rows, CONTACT_COLS, figsize=(7.0 * CONTACT_COLS, 3.1 * n_rows), squeeze=False)
        for ax in axes.ravel():
            ax.set_axis_off()
        for ax, (p, r) in zip(axes.ravel(), chunk):
            ax.imshow(plt.imread(p))
            extra = f"  e->bc={int(r['n_exp_to_bc'])}" if "n_exp_to_bc" in r else ""
            ax.set_title(f"{r['content']} #{int(r['rank']):03d}  switches={r['n_switches']}{extra}  {r['unique_id']}", fontsize=8)
        fig.suptitle(f"{mol} — page {page + 1}/{n_pages} ({len(paths)} movies)", fontsize=13)
        fig.tight_layout()
        sheet = BATCH_ROOT / f"contact_sheet_{mol}_p{page + 1:02d}.png"
        fig.savefig(sheet, dpi=110, facecolor="white")
        if CONTACT_SHOW_INLINE:
            plt.show()
        else:
            plt.close(fig)
        print("Saved:", sheet)

In [ ]:
# =============================================================================
# Collect uids from a folder of hand-picked movies
# =============================================================================
# Expects the batch layout: <PICKED_DIR>/<mol>/<content>_<rank>_<unique_id>/
PICKED_DIR = MOVIE_DIR / "FINALPICKED"

picked_rows = []
for folder in sorted(p for p in PICKED_DIR.rglob("*") if p.is_dir() and "uid_" in p.name):
    prefix, tail = folder.name.split("uid_", 1)
    uid = "uid_" + tail
    win_csv = folder / "windows.csv"
    if win_csv.exists():
        ids = pd.read_csv(win_csv, usecols=["unique_id"])["unique_id"].astype(str).unique()
        if len(ids) == 1:
            uid = ids[0]
    parts = prefix.strip("_").split("_")
    picked_rows.append({
        "mol_folder": folder.parent.name,
        "content": parts[0] if parts and parts[0] else "picked",
        "rank": int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0,
        "unique_id": uid,
        "folder": str(folder),
    })

picked_df = pd.DataFrame(picked_rows).drop_duplicates("unique_id").reset_index(drop=True)
PICKED_UIDS = picked_df["unique_id"].tolist()
display(picked_df[["mol_folder", "content", "rank", "unique_id"]])
print(f"{len(PICKED_UIDS)} uids")
print(PICKED_UIDS)

In [ ]:
# =============================================================================
# Re-render picked uids with custom settings — EDIT EVERYTHING HERE
# =============================================================================
RERENDER_UIDS = PICKED_UIDS            # or a hand-written list, or PICKED_UIDS[:1] to test one
RERENDER_OUT_DIR = PICKED_DIR / "rerendered4"
RERENDER_TAG = "v2"                    # subfolder per settings version, so versions don't overwrite
RERENDER_SKIP_EXISTING = False
RERENDER_SHOW_TITLE = True
RERENDER_WRITE_SVG = True              # final frame as SVG for Illustrator
RERENDER_WRITE_GIF = False
RERENDER_SHOW_STILLS = True            # display each final frame inline

VID_SETTINGS = dict(
    # --- playback ---
    MOVIE_SPEED=0.35,                  # vs real time (1.0 = real time)
    MOVIE_FPS=30,
    MOVIE_DPI=150,
    MOVIE_FIGSIZE=(11.2, 4.8),         # inches; figsize * dpi must be even in both directions
    LEAD_IN_S=0.4,                     # movie seconds held on t = 0
    TAIL_HOLD_S=1.2,                   # movie seconds held on the last frame
    # --- smoothing / timing ---
    SOFTMAX_SMOOTH_N=1500,
    SOFTMAX_ANCHOR="center",           # "center" | "start"
    FRAME_STATE_POLICY="nearest_center",  # "nearest_center" | "later_window" | "earlier_window"
    TRACK_SMOOTH=1.2,                  # Gaussian sigma in frames; 0 = off
    TRACK_CURVE=4,                     # cubic samples per frame step; 1 = straight chords
    TRACK_LINEWIDTH=2.4,
    # --- XY panel ---
    XY_SPAN_UM=8,                   # None = fit track; number = fixed square box (um)
    XY_PAD_FRAC=0.12,
    XY_INVERT_Y=False,
    SCALE_BAR_UM=1,
    HEAD_SIZE_PT=7.0,
    # --- state-change flash ---
    FLASH_ON_CHANGE=True,
    FLASH_DURATION_S=0.55,
    FREEZE_ON_CHANGE_S=0.0,            # movie seconds the clock pauses at each change
    FLASH_GOLD=(1.0, 0.84, 0.15),
    HEAD_PULSE_SCALE=1.0,
    # --- softmax panel ---
    SHOW_ASSIGNMENT_RIBBON=True,
    SHOW_FUTURE_GHOST=False,
    MARK_CHANGES_ON_AREA=True,
    # --- raw-data panel (leftmost when on; off = same layout as before) ---
    RAW_PANEL=True,
    RAW_PANEL_WIDTH_IN=4.2,            # figure width added for the panel
    RAW_ON_MISSING="skip_panel",       # "skip_panel" | "error"
    RAW_MASTER_DIR=None,               # override experiment folder holding data/<condition>/<file>.tif
    RAW_PATH_OVERRIDES={},             # {unique_id: r"path/to/movie.tif"}
    #   LUT
    RAW_CMAP="gray",                   # matplotlib cmap name: "gray" | "magma" | "inferno" | "viridis" | ...
    RAW_USE_FIJI_LUT=False,
    RAW_FIJI_LUT_DIR=str(paths["fiji_lut_dir"]),
    RAW_FIJI_LUT_NAME="JDM_Ink Wash Yellow",  # list_fiji_luts("JDM") shows the options
    RAW_REVERSE_LUT=False,
    RAW_INVERT=False,                  # dark spots on light background
    #   contrast
    RAW_CONTRAST_LIMITS=None,          # (lo, hi) raw counts; None = percentiles
    RAW_CONTRAST_PERCENTILES=(1.0, 99.7),
    RAW_CONTRAST_PER_FRAME=False,
    RAW_GAMMA=1.0,
    RAW_TEMPORAL_AVG=1,                # rolling mean over N raw frames
    RAW_FRAME_BLEND=False,
    RAW_INTERPOLATION="nearest",       # "nearest" | "bilinear" | "bicubic"
    #   field of view
    RAW_FOV_MODE="track_crop",         # "track_crop" | "follow"
    RAW_SPAN_UM="match_xy",            # "match_xy" | number (um) | None = fit track + RAW_PAD_UM
    RAW_PAD_UM=0.5,
    RAW_FOLLOW_BOX_UM=3.0,
    RAW_FORCE_XY_IMAGE_ORIENTATION=True,  # XY panel y-down so it matches the image
    #   track overlay
    RAW_SHOW_TRACK=True,
    RAW_TRACK_LINEWIDTH=1.6,
    RAW_TRACK_ALPHA=0.85,
    RAW_TRAIL_MODE="cumulative",       # "cumulative" | "tail"
    RAW_TAIL_S=0.5,
    RAW_TAIL_FADE=True,
    RAW_HEAD_STYLE="ring",             # "ring" | "dot" | "none"
    RAW_HEAD_SIZE_PT=10.0,
    RAW_FLASH_GLOW=True,
    RAW_FLASH_WHITEN=0.0,              # 0-1 white lift of the image at state changes
    RAW_SCALE_BAR_UM=1.0,
    RAW_TEXT_COLOR="auto",
    RAW_SHOW_TIMER=False,
)

# -----------------------------------------------------------------------------
import json

_w_px = VID_SETTINGS["MOVIE_FIGSIZE"][0] * VID_SETTINGS["MOVIE_DPI"]
_h_px = VID_SETTINGS["MOVIE_FIGSIZE"][1] * VID_SETTINGS["MOVIE_DPI"]
if round(_w_px) % 2 or round(_h_px) % 2:
    raise ValueError(f"MOVIE_FIGSIZE * MOVIE_DPI = {_w_px:g} x {_h_px:g} px; both must be even for H.264")

rerender_root = Path(RERENDER_OUT_DIR) / RERENDER_TAG
rerender_root.mkdir(parents=True, exist_ok=True)
(rerender_root / "settings.json").write_text(json.dumps({k: v for k, v in VID_SETTINGS.items()}, indent=1, default=str))

picked_meta = picked_df.set_index("unique_id").to_dict("index") if "picked_df" in globals() else {}
uid_list = [str(u) for u in RERENDER_UIDS]
windowed_rr, _ = apply_dataset_filter(
    windowed_full.filter(pl.col("unique_id").cast(pl.Utf8).is_in(uid_list)), {}, use_extra=False
)

_UNSET = object()
_saved_settings = {k: globals().get(k, _UNSET) for k in VID_SETTINGS}
globals().update(VID_SETTINGS)
rerender_log = []
t_all = time.time()
try:
    for n_i, uid in enumerate(uid_list, start=1):
        meta_p = picked_meta.get(uid, {})
        t0 = time.time()
        rec = {"unique_id": uid, "status": "", "error": ""}
        try:
            seq_r = (
                windowed_rr.filter(pl.col("unique_id").cast(pl.Utf8) == uid)
                .sort("time_window").to_pandas().reset_index(drop=True)
            )
            if seq_r.empty:
                raise ValueError("uid not in the windowed table (or all windows ungated)")
            mol = str(seq_r["mol"].iloc[0]) if "mol" in seq_r.columns else meta_p.get("mol_folder", "unknown")
            content, rank = meta_p.get("content", "picked"), int(meta_p.get("rank", 0))
            out_dir = rerender_root / mol / f"{content}_{rank:02d}_{uid}"
            mp4 = out_dir / "live_state_softmax.mp4"
            rec.update(mol=mol, content=content, rank=rank, mp4=str(mp4))
            if RERENDER_SKIP_EXISTING and mp4.exists():
                rec["status"] = "skipped_existing"
            else:
                out_dir.mkdir(parents=True, exist_ok=True)
                inst_r = load_track_instant(uid)
                data_r = build_movie_data(seq_r, inst_r)
                ta_r, fw_r, fl_r = build_timeline(data_r)
                title_r = None
                if RERENDER_SHOW_TITLE:
                    m0 = seq_r.iloc[0]
                    title_r = "   ".join([
                        f"{content} #{rank:02d}", uid,
                        *[f"{c}={m0[c]}" for c in ("mol", "geno", "type", "location") if c in seq_r.columns],
                        f"{len(seq_r)} windows",
                    ])
                live_r = LiveStateSoftmaxMovie(data_r, ta_r, fw_r, fl_r, seq_r, title=title_r, inst=inst_r)
                rec.update(raw_path=live_r.raw_path, raw_error=live_r.raw_error)
                try:
                    live_r.save_still(len(ta_r) - 1, out_dir / "final_frame.png")
                    if RERENDER_WRITE_SVG:
                        live_r.save_still(len(ta_r) - 1, out_dir / "final_frame.svg")
                    live_r.save_mp4(mp4)
                    if RERENDER_WRITE_GIF:
                        live_r.save_gif(out_dir / "live_state_softmax.gif")
                finally:
                    plt.close(live_r.fig)
                seq_r.to_csv(out_dir / "windows.csv", index=False)
                rec.update(status="ok", movie_frames=len(ta_r), n_changes=len(data_r["changes"]))
                if RERENDER_SHOW_STILLS:
                    display(Image(filename=str(out_dir / "final_frame.png"), width=800))
        except Exception as exc:
            rec.update(status="failed", error=f"{type(exc).__name__}: {exc}")
            plt.close("all")
        rec["render_s"] = round(time.time() - t0, 1)
        rerender_log.append(rec)
        pd.DataFrame(rerender_log).to_csv(rerender_root / "rerender_manifest.csv", index=False)
        print(f"[{n_i}/{len(uid_list)}] {uid}: {rec['status']} ({rec['render_s']} s)"
              + (f"  {rec['error']}" if rec["error"] else ""))
finally:
    for _k, _v in _saved_settings.items():
        if _v is _UNSET:
            globals().pop(_k, None)
        else:
            globals()[_k] = _v

print(f"\nDone in {(time.time() - t_all) / 60:.1f} min -> {rerender_root}")
display(pd.DataFrame(rerender_log))

In [ ]:
# =============================================================================
# Supplementary movies (Molecular Cell spec) — one MP4 per molecule, its picked tracks played in sequence
# =============================================================================
import json
import subprocess

import imageio_ffmpeg

SUPP_PICK_DIR = (
    paths["paper_plots_dir"] / "FiguresRound25_PACKAGEFORCO-AUTHORS" / "potentialvids" / "FINALPICKED" / "rerendered4" / "v2"
)
SUPP_EXTRA = {"myosin": ["mixed_02_uid_GCcmrw_p27371_rnone_b66ed1fd6d"]}  # folder-style names or bare uids
SUPP_ORDER = {}                     # {"myosin": [uid, uid, uid]} to set play order; default = folder-name order
SUPP_MOLECULES = ["HTT", "kinesin", "myosin"]
SUPP_MOL_LABELS = {"HTT": "HTT", "kinesin": "Kinesin", "myosin": "Myosin"}
SUPP_OUT_DIR = SUPP_PICK_DIR.parent.parent / "supp_movies_molcell_v2"
SUPP_FILE_STEM = "SuppMovie_{mol}"

# Molecular Cell: H.264 .mp4, 30 fps, 16:9 (or 4:3), >= 320x240, 2 Mbps video, AAC 128 kbps audio, < 150 MB,
# text legible when shown 320 px wide -> fonts are sized for a 4x downscale of 1280x720.
SUPP_SIZE_PX = (1280, 720)
SUPP_DPI = 100
SUPP_VIDEO_KBPS = 2000
SUPP_AUDIO = True                   # silent AAC stereo track at 128 kbps
SUPP_TITLE_CARD_S = 1.5             # card before each track; 0 = none
SUPP_CARD_META = False              # cell type / genotype line on the card
SUPP_FONT = dict(header=26, title=20, label=18, tick=17, legend=18, state=24, bar=18, card=46, card_sub=30)
SUPP_KEEP_SEGMENTS = False          # keep the per-track intermediate MP4s

SUPP_SETTINGS = json.loads((SUPP_PICK_DIR / "settings.json").read_text())
SUPP_SETTINGS.update(
    MOVIE_FPS=30, MOVIE_DPI=SUPP_DPI, MOVIE_FIGSIZE=(SUPP_SIZE_PX[0] / SUPP_DPI, SUPP_SIZE_PX[1] / SUPP_DPI),
    RAW_PANEL=True, RAW_ON_MISSING="error",
    RAW_CMAP="gray", RAW_USE_FIJI_LUT=False, RAW_REVERSE_LUT=False, RAW_INVERT=True,  # white bg, black molecules
    TRACK_LINEWIDTH=3.4, HEAD_SIZE_PT=12.0, RAW_TRACK_LINEWIDTH=2.6, RAW_HEAD_SIZE_PT=18.0,
)

# -----------------------------------------------------------------------------
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()


def _supp_label(s):
    return str(s).replace("unresolved_", "unresolved ").replace("_", " + ")


class SuppStateMovie(LiveStateSoftmaxMovie):
    """16:9 layout: raw | track | state key on top, softmax + assignment ribbon full width below."""

    def __init__(self, data, taus, flash_w, flash_lab, seq, inst, header):
        self.d, self.taus, self.flash_w, self.flash_lab = data, taus, flash_w, flash_lab
        self.n_frames = len(taus)
        observed = mixture_state_order(data["states"])
        self.colors = {s: mixture_state_color(s) for s in set(observed) | set(PURE_STATES)}
        self.gold = np.array(FLASH_GOLD, dtype=float)
        self.raw = prepare_raw_panel(seq, inst, data)
        self.raw_path, self.raw_error = str(self.raw["path"]), None
        self.xy_invert = True

        fw_in, fh_in = MOVIE_FIGSIZE
        fig = plt.figure(figsize=(fw_in, fh_in), dpi=MOVIE_DPI)
        fig.patch.set_facecolor("white")
        sq_b, sq_h = 0.41, 0.45
        sq_w = sq_h * fh_in / fw_in
        self.ax_raw = fig.add_axes([0.03, sq_b, sq_w, sq_h])
        self.ax_xy = fig.add_axes([0.06 + sq_w, sq_b, sq_w, sq_h])
        self.ax_leg = fig.add_axes([0.10 + 2 * sq_w, sq_b, 0.97 - (0.10 + 2 * sq_w), sq_h])
        self.ax_soft = fig.add_axes([0.085, 0.215, 0.885, 0.16])
        self.ax_rib = fig.add_axes([0.085, 0.155, 0.885, 0.045], sharex=self.ax_soft)
        self.fig = fig

        self._setup_xy()
        self._setup_raw()
        self._setup_soft()
        self._setup_ribbon()
        self._restyle(observed, header)
        self.draw(0)

    def _restyle(self, observed, header):
        F = SUPP_FONT
        ax = self.ax_xy
        ax.set_title("Track (coloured by state)", fontsize=F["title"])
        for t in ax.texts:
            if "\u00b5m" in t.get_text():
                t.set_fontsize(F["bar"])
                t.set_va("bottom")
        for ln in ax.lines:
            ln.set_linewidth(3.5)
        self.txt_time.set_visible(False)
        self.txt_state.set_visible(False)

        ax = self.ax_raw
        ax.set_title("Raw data + track", fontsize=F["title"])
        for a in ax.artists:
            if isinstance(a, AnchoredSizeBar):
                a.txt_label._text.set_fontsize(F["bar"])
        self.raw_head.set_linewidths([3.0])

        ax = self.ax_soft
        ax.set_title("")
        ax.set_ylabel("P(state)", fontsize=F["label"])
        ax.set_yticks([0, 0.5, 1])
        ax.tick_params(labelsize=F["tick"], width=1.5, length=6)
        self.tips.set_sizes([110] * len(PURE_STATES))
        self.cursor.set_linewidth(2.0)
        ax = self.ax_rib
        ax.set_xlabel("Time (s)", fontsize=F["label"])
        ax.tick_params(labelsize=F["tick"], width=1.5, length=6)
        self.rib_cursor.set_linewidth(2.0)

        ax = self.ax_leg
        ax.axis("off")
        ax.text(0.0, 1.0, "State", transform=ax.transAxes, fontsize=F["title"], fontweight="bold", va="top")
        self.now_txt = ax.text(0.0, 0.87, "", transform=ax.transAxes, fontsize=F["state"],
                               fontweight="bold", va="top")
        keys = list(PURE_STATES) + [s for s in observed if s not in PURE_STATES]
        ax.legend(handles=[Patch(facecolor=self.colors[s], label=_supp_label(s)) for s in keys],
                  loc="upper left", bbox_to_anchor=(0.0, 0.70), frameon=False, fontsize=F["legend"],
                  handlelength=1.2, handleheight=0.9, labelspacing=0.3, borderaxespad=0.0, borderpad=0.0)

        self.fig.text(0.03, 0.945, header, fontsize=F["header"], fontweight="bold", va="center")
        self.hdr_time = self.fig.text(0.97, 0.945, "", ha="right", va="center",
                                      fontsize=F["header"] - 2, family="monospace")

    def draw(self, k):
        super().draw(k)
        tau, w = float(self.taus[k]), float(self.flash_w[k])
        state = self._state_at(tau)
        lab = self.flash_lab[k] if (w > 0 and self.flash_lab[k] is not None) else state
        self.now_txt.set_text(_supp_label(lab))
        self.now_txt.set_color(self.colors.get(state, "#999999"))
        self.hdr_time.set_text(f"t = {tau:4.2f} s")
        return []


def _title_card(path, title, sub, meta):
    F = SUPP_FONT
    fig = plt.figure(figsize=MOVIE_FIGSIZE, dpi=MOVIE_DPI)
    fig.patch.set_facecolor("white")
    fig.text(0.5, 0.58, title, ha="center", va="center", fontsize=F["card"], fontweight="bold")
    fig.text(0.5, 0.44, sub, ha="center", va="center", fontsize=F["card_sub"])
    if meta:
        fig.text(0.5, 0.33, meta, ha="center", va="center", fontsize=F["label"], color="0.35")
    fig.savefig(path, dpi=MOVIE_DPI, facecolor="white")
    plt.close(fig)


def _ffmpeg(args):
    res = subprocess.run([FFMPEG, "-hide_banner", "-loglevel", "error", "-y", *map(str, args)],
                         capture_output=True, text=True)
    if res.returncode:
        raise RuntimeError(f"ffmpeg failed: {res.stderr[-1500:]}")


def _probe(path):
    res = subprocess.run([FFMPEG, "-hide_banner", "-i", str(path)], capture_output=True, text=True)
    return [ln.strip() for ln in res.stderr.splitlines() if ln.strip().startswith(("Duration", "Stream"))]


def _uid_of(name):
    name = str(name)
    return name[name.index("uid_"):] if "uid_" in name else name


supp_tracks = {}
for mol in SUPP_MOLECULES:
    names = sorted(p.name for p in (SUPP_PICK_DIR / mol).iterdir() if p.is_dir() and "uid_" in p.name)
    names += [n for n in SUPP_EXTRA.get(mol, []) if _uid_of(n) not in map(_uid_of, names)]
    uids = [_uid_of(n) for n in names]
    if SUPP_ORDER.get(mol):
        uids = [_uid_of(u) for u in SUPP_ORDER[mol]]
    supp_tracks[mol] = uids
    print(f"{mol}: {uids}")

_UNSET = object()
_saved_supp = {k: globals().get(k, _UNSET) for k in SUPP_SETTINGS}
globals().update(SUPP_SETTINGS)
SUPP_OUT_DIR.mkdir(parents=True, exist_ok=True)
(SUPP_OUT_DIR / "settings.json").write_text(json.dumps(
    {**SUPP_SETTINGS, "SUPP_SIZE_PX": SUPP_SIZE_PX, "SUPP_VIDEO_KBPS": SUPP_VIDEO_KBPS, "SUPP_FONT": SUPP_FONT,
     "SUPP_TITLE_CARD_S": SUPP_TITLE_CARD_S, "tracks": supp_tracks}, indent=1, default=str))
all_uids = [u for us in supp_tracks.values() for u in us]
windowed_supp, _ = apply_dataset_filter(
    windowed_full.filter(pl.col("unique_id").cast(pl.Utf8).is_in(all_uids)), {}, use_extra=False
)
supp_log = []
t_all = time.time()
try:
    for mol in SUPP_MOLECULES:
        uids = supp_tracks[mol]
        mol_dir = SUPP_OUT_DIR / mol
        seg_dir = mol_dir / "_segments"
        seg_dir.mkdir(parents=True, exist_ok=True)
        segments, t_cursor = [], 0.0
        for i, uid in enumerate(uids, start=1):
            t0 = time.time()
            seq_s = (windowed_supp.filter(pl.col("unique_id").cast(pl.Utf8) == uid)
                     .sort("time_window").to_pandas().reset_index(drop=True))
            if seq_s.empty:
                raise ValueError(f"{uid} not in the windowed table")
            m0 = seq_s.iloc[0]
            label = SUPP_MOL_LABELS.get(mol, mol)
            if SUPP_TITLE_CARD_S > 0:
                meta = "   ".join(f"{n}: {m0[c]}" for c, n in (("type", "cell type"), ("geno", "genotype"))
                                  if c in seq_s.columns) if SUPP_CARD_META else ""
                png = seg_dir / f"{i:02d}_card.png"
                _title_card(png, label, f"Track {i} of {len(uids)}", meta)
                card = seg_dir / f"{i:02d}_card.mp4"
                _ffmpeg(["-loop", 1, "-framerate", MOVIE_FPS, "-i", png, "-t", SUPP_TITLE_CARD_S,
                         "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", 12, "-r", MOVIE_FPS, card])
                segments.append(card)
                t_cursor += SUPP_TITLE_CARD_S
            inst_s = load_track_instant(uid)
            data_s = build_movie_data(seq_s, inst_s)
            ta_s, fw_s, fl_s = build_timeline(data_s)
            live_s = SuppStateMovie(data_s, ta_s, fw_s, fl_s, seq_s, inst_s,
                                    header=f"{label}   \u00b7   track {i} of {len(uids)}")
            seg = seg_dir / f"{i:02d}_{uid}.mp4"
            try:
                live_s.save_still(len(ta_s) - 1, mol_dir / f"track{i}_{uid}_final.png")
                live_s.save_still(len(ta_s) - 1, mol_dir / f"track{i}_{uid}_final.svg")
                live_s.save_mp4(seg, crf=12)
            finally:
                plt.close(live_s.fig)
            segments.append(seg)
            dur = live_s.n_frames / MOVIE_FPS
            supp_log.append(dict(mol=mol, order=i, unique_id=uid, starts_at_s=round(t_cursor, 2),
                                 duration_s=round(dur, 2), n_changes=len(data_s["changes"]),
                                 raw_path=live_s.raw_path, render_s=round(time.time() - t0, 1),
                                 **{c: m0[c] for c in ("geno", "type", "location", "filename") if c in seq_s.columns}))
            t_cursor += dur

        out = SUPP_OUT_DIR / (SUPP_FILE_STEM.format(mol=mol) + ".mp4")
        lst = seg_dir / "concat.txt"
        lst.write_text("".join(f"file '{p.as_posix()}'\n" for p in segments))
        args = ["-f", "concat", "-safe", 0, "-i", lst]
        if SUPP_AUDIO:
            args += ["-f", "lavfi", "-i", "anullsrc=channel_layout=stereo:sample_rate=48000"]
        args += ["-map", "0:v:0"] + (["-map", "1:a:0"] if SUPP_AUDIO else [])
        args += ["-c:v", "libx264", "-profile:v", "high", "-pix_fmt", "yuv420p", "-r", MOVIE_FPS,
                 "-b:v", f"{SUPP_VIDEO_KBPS}k", "-maxrate", f"{SUPP_VIDEO_KBPS}k", "-bufsize", f"{2 * SUPP_VIDEO_KBPS}k",
                 "-preset", "slow", "-x264-params", f"keyint={2 * MOVIE_FPS}"]
        if SUPP_AUDIO:
            args += ["-c:a", "aac", "-b:a", "128k", "-shortest"]
        args += ["-movflags", "+faststart", out]
        _ffmpeg(args)

        for r in supp_log:
            if r["mol"] == mol:
                p320 = mol_dir / f"preview320_track{r['order']}.png"
                _ffmpeg(["-ss", r["starts_at_s"] + 0.8 * r["duration_s"], "-i", out,
                         "-vf", "scale=320:-2:flags=area", "-frames:v", 1, p320])
        if not SUPP_KEEP_SEGMENTS:
            for p in seg_dir.iterdir():
                p.unlink()
            seg_dir.rmdir()
        print(f"\n{out}  {out.stat().st_size / 1e6:.1f} MB")
        for ln in _probe(out):
            print("   ", ln)
finally:
    for _k, _v in _saved_supp.items():
        if _v is _UNSET:
            globals().pop(_k, None)
        else:
            globals()[_k] = _v

supp_df = pd.DataFrame(supp_log)
supp_df.to_csv(SUPP_OUT_DIR / "supp_movies_manifest.csv", index=False)
print(f"\nDone in {(time.time() - t_all) / 60:.1f} min -> {SUPP_OUT_DIR}")
display(supp_df)
for mol in SUPP_MOLECULES:
    p = sorted((SUPP_OUT_DIR / mol).glob("track1_*_final.png"))
    if p:
        display(Image(filename=str(p[0]), width=900))